User Verification  
• Input: User’s voice.  
• Task: Determine if user is who they say they are.   
• Input State:  VA is “locked”.   
• Output State: Once the User is verified or identified, the system goes into a “Unlocked” state   
waiting to be awaken for further instructions. If not, the system stays in “Locked” state.   
• By-pass: Allow the user to enter a code to unlock the system.

We will use both Resemblyzer, which will conduct the main text-independent user verification, anbd MFCC with librosa as a secondary check and baseline

In [ ]:
!pip -q install resemblyzer librosa soundfile pydub numpy scipy pandas
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 21.6 MB/s eta 0:00:00


In [ ]:
#Grab all the audio of the registered users
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive')
ENROLLED_DIR = "/content/drive/MyDrive/project_dataset_audio/enrolled_verification"

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
# Import the libraries that allow for speech processing and speaker verification

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
import tempfile
import warnings
from IPython.display import Audio
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from pathlib import Path
from collections import defaultdict
from pydub import AudioSegment
from resemblyzer import VoiceEncoder, preprocess_wav
import tensorflow as tf
from tensorflow.keras import layers, models
import soundfile as sf

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# Silence some warning messages
warnings.filterwarnings(
    "ignore",
    message="PySoundFile failed.*",
    category=UserWarning
)
warnings.filterwarnings(
    "ignore",
    message="librosa.core.audio.__audioread_load",
    category=FutureWarning
)
warnings.filterwarnings("ignore", message="audioread.*")

In [ ]:
# list all available enrolled files

for f in sorted(os.listdir(ENROLLED_DIR)):
    print(f)

Barriere-near-1.m4a
Barriere-near-2.m4a
Barriere-near-3.m4a
Barriere-near-4.m4a
Barriere-other-1.m4a
Barriere-other-2.m4a
Barriere-other-3.m4a
Barriere-other-4.m4a
Barriere-positive-1.m4a
Barriere-positive-2.m4a
Barriere-positive-3.m4a
Barriere-positive-4.m4a
Cherukunnummal Poothakandi-near-1.m4a
Cherukunnummal Poothakandi-near-2.m4a
Cherukunnummal Poothakandi-near-3.m4a
Cherukunnummal Poothakandi-near-4.m4a
Cherukunnummal Poothakandi-other-1.m4a
Cherukunnummal Poothakandi-other-2.m4a
Cherukunnummal Poothakandi-other-3.m4a
Cherukunnummal Poothakandi-other-4.m4a
Cherukunnummal Poothakandi-positive-1.m4a
Cherukunnummal Poothakandi-positive-2.m4a
Cherukunnummal Poothakandi-positive-3.m4a
Cherukunnummal Poothakandi-positive-4.m4a
Drishya_text.m4a
Drishya_text10.m4a
Drishya_text11.m4a
Drishya_text12.m4a
Drishya_text13.m4a
Drishya_text14.m4a
Drishya_text15.m4a
Drishya_text16.m4a
Drishya_text17.m4a
Drishya_text18.m4a
Drishya_text19.m4a
Drishya_text2.m4a
Drishya_text20.m4a
Drishya_text3.m4a
Dr

In [ ]:
# Parameters for preprocessing and MFCC extraction

TARGET_SR = 16000

N_MFCC = 13
WINDOW_SEC = 0.025
HOP_SEC = 0.025

# Thresholds for the verification decision
# You may need to tune these after testing on your team's audio
EMBED_THRESHOLD = 0.90
MFCC_THRESHOLD = 0.85
FINAL_THRESHOLD = 0.89

# Give more importance to the Resemblyzer embedding score
EMBED_WEIGHT = 0.80
MFCC_WEIGHT = 0.20

# PIN used for bypass
BYPASS_PIN = "1234"

In [ ]:
# Helper functions

# Map the names so that they fall under the same names:
NAME_MAP = {
    "Salman": "Farina",
    "Gaffar": "Ziyad",
    "Barriere": "Caroline",
    "Cherukunnummal Poothakandi": "Drishya"
}

# Extract speaker name
def get_speaker_name(filename):
    return Path(filename).stem.split("-")[0]

# Normalize the speaker
def normalize_speaker(name):
    return NAME_MAP.get(name, name)

# Convert File to .wav
def convert_to_wav(file_path, target_sr=TARGET_SR):
    suffix = Path(file_path).suffix.lower()

    if suffix == ".wav":
        return file_path

    audio = AudioSegment.from_file(file_path)
    audio = audio.set_channels(1).set_frame_rate(target_sr)

    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    tmp_path = tmp.name
    tmp.close()

    audio.export(tmp_path, format="wav")

    return tmp_path

# Group all files with same speaker
SUPPORTED_EXTENSIONS = (".m4a", ".wav")
speaker_files = defaultdict(list)

for filename in sorted(os.listdir(ENROLLED_DIR)):
    if filename.lower().endswith(SUPPORTED_EXTENSIONS):
        raw_speaker = get_speaker_name(filename)
        speaker = normalize_speaker(raw_speaker)

        full_path = os.path.join(ENROLLED_DIR, filename)
        speaker_files[speaker].append(full_path)

for speaker, files in speaker_files.items():
    print(f"\n{speaker}")
    for f in files:
        print("  ", os.path.basename(f))


Caroline
   Barriere-near-1.m4a
   Barriere-near-2.m4a
   Barriere-near-3.m4a
   Barriere-near-4.m4a
   Barriere-other-1.m4a
   Barriere-other-2.m4a
   Barriere-other-3.m4a
   Barriere-other-4.m4a
   Barriere-positive-1.m4a
   Barriere-positive-2.m4a
   Barriere-positive-3.m4a
   Barriere-positive-4.m4a

Drishya
   Cherukunnummal Poothakandi-near-1.m4a
   Cherukunnummal Poothakandi-near-2.m4a
   Cherukunnummal Poothakandi-near-3.m4a
   Cherukunnummal Poothakandi-near-4.m4a
   Cherukunnummal Poothakandi-other-1.m4a
   Cherukunnummal Poothakandi-other-2.m4a
   Cherukunnummal Poothakandi-other-3.m4a
   Cherukunnummal Poothakandi-other-4.m4a
   Cherukunnummal Poothakandi-positive-1.m4a
   Cherukunnummal Poothakandi-positive-2.m4a
   Cherukunnummal Poothakandi-positive-3.m4a
   Cherukunnummal Poothakandi-positive-4.m4a

Drishya_text
   Drishya_text.m4a

Drishya_text10
   Drishya_text10.m4a

Drishya_text11
   Drishya_text11.m4a

Drishya_text12
   Drishya_text12.m4a

Drishya_text13
   Drishy

In [ ]:
# Functions for the User Verification

# Define the extract_mfcc with default parameters
def extract_mfcc(file_path, sr=TARGET_SR, window_sec=WINDOW_SEC, hop_sec=HOP_SEC, n_mfcc=N_MFCC):
    """
    file_path: path to audio file
    sr: sampling rate
    window_sec: window size in seconds
    hop_sec: hop size in seconds
    n_mfcc: number of MFCC coefficients
    """

    wav_path = convert_to_wav(file_path, target_sr=sr)

    # Load audio
    y, sr = librosa.load(wav_path, sr=sr, mono=True)

    n_fft = int(window_sec * sr)
    hop_length = int(hop_sec * sr)

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length
    )

    return mfcc

# Since MFCC outputs a matrix over time, reduce it to a fixed-size vector by taking mean and standard deviation over time
def extract_mfcc_signature(file_path):
    mfcc = extract_mfcc(file_path)

    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    signature = np.concatenate([mfcc_mean, mfcc_std], axis=0)
    return signature.astype(np.float32)


# Load the Resemblyzer encoder once
encoder = VoiceEncoder()

# Extract speaker embedding from one audio file
def extract_embedding(file_path):
    wav_path = convert_to_wav(file_path, target_sr=TARGET_SR)
    wav = preprocess_wav(wav_path)
    embedding = encoder.embed_utterance(wav)
    return np.array(embedding, dtype=np.float32)

Loaded the voice encoder model on cpu in 0.22 seconds.


In [ ]:
# Compare 2 vectors with cosine similarity
def cosine_similarity(a, b):
    a = np.asarray(a).flatten()
    b = np.asarray(b).flatten()

    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-10
    return float(np.dot(a, b) / denom)

def normalize_cosine(score):
    # Convert cosine similarity from [-1, 1] to [0, 1]
    return (score + 1.0) / 2.0

In [ ]:
# For each speaker:
# - average all Resemblyzer embeddings
# - average all MFCC signatures
profiles = {}
for speaker, files in speaker_files.items():
    emb_list = []
    mfcc_list = []

    for file_path in files:
        try:
            emb = extract_embedding(file_path)
            mfcc_sig = extract_mfcc_signature(file_path)

            emb_list.append(emb)
            mfcc_list.append(mfcc_sig)

        except Exception as e:
            print("Skipped:", file_path, "|", e)

    if len(emb_list) == 0:
        print("Could not build profile for", speaker)
        continue

    profiles[speaker] = {
        "embedding_profile": np.mean(np.stack(emb_list), axis=0),
        "mfcc_profile": np.mean(np.stack(mfcc_list), axis=0),
        "num_files": len(emb_list)
    }

print("Profiles created for:")
for speaker in profiles:
    print(" ", speaker, "-", profiles[speaker]["num_files"], "files")

Profiles created for:
  Caroline - 12 files
  Drishya - 12 files
  Drishya_text - 1 files
  Drishya_text10 - 1 files
  Drishya_text11 - 1 files
  Drishya_text12 - 1 files
  Drishya_text13 - 1 files
  Drishya_text14 - 1 files
  Drishya_text15 - 1 files
  Drishya_text16 - 1 files
  Drishya_text17 - 1 files
  Drishya_text18 - 1 files
  Drishya_text19 - 1 files
  Drishya_text2 - 1 files
  Drishya_text20 - 1 files
  Drishya_text3 - 1 files
  Drishya_text4 - 1 files
  Drishya_text5 - 1 files
  Drishya_text6 - 1 files
  Drishya_text7 - 1 files
  Drishya_text8 - 1 files
  Drishya_text9 - 1 files
  Farina - 22 files
  Ziyad - 22 files


In [ ]:
# Example: inspect one enrolled speaker
example_speaker = list(profiles.keys())[2]
print("Speaker:", example_speaker)
print("Embedding profile shape:", profiles[example_speaker]["embedding_profile"].shape)
print("MFCC profile shape:", profiles[example_speaker]["mfcc_profile"].shape)

Speaker: Drishya_text
Embedding profile shape: (256,)
MFCC profile shape: (26,)


In [ ]:
# Verify if test audio matches ANY enrolled user

def verify_any_user(test_file, pin=None):
    state_before = "Locked"

    # PIN bypass
    if pin is not None and pin == BYPASS_PIN:
        return {
            "matched_user": None,
            "best_embedding_score": 0.0,
            "best_mfcc_score": 0.0,
            "best_final_score": 0.0,
            "accepted": True,
            "state_before": state_before,
            "state_after": "Unlocked",
            "bypass_used": True
        }

    test_embedding = extract_embedding(test_file)
    test_mfcc = extract_mfcc_signature(test_file)

    best_user = None
    best_embedding_score = -1
    best_mfcc_score = -1
    best_final_score = -1

    # Compare the test voice to all enrolled users
    for user, data in profiles.items():
        enrolled_embedding = data["embedding_profile"]
        enrolled_mfcc = data["mfcc_profile"]

        raw_embedding_score = cosine_similarity(enrolled_embedding, test_embedding)
        raw_mfcc_score = cosine_similarity(enrolled_mfcc, test_mfcc)

        embedding_score = normalize_cosine(raw_embedding_score)
        mfcc_score = normalize_cosine(raw_mfcc_score)

        final_score = EMBED_WEIGHT * embedding_score + MFCC_WEIGHT * mfcc_score

        if final_score > best_final_score:
            best_final_score = final_score
            best_embedding_score = embedding_score
            best_mfcc_score = mfcc_score
            best_user = user

    accepted = (
        best_embedding_score >= EMBED_THRESHOLD and
        best_mfcc_score >= MFCC_THRESHOLD and
        best_final_score >= FINAL_THRESHOLD
    )

    state_after = "Unlocked" if accepted else "Locked"

    return {
        "matched_user": best_user,
        "best_embedding_score": float(best_embedding_score),
        "best_mfcc_score": float(best_mfcc_score),
        "best_final_score": float(best_final_score),
        "accepted": bool(accepted),
        "state_before": state_before,
        "state_after": state_after,
        "bypass_used": False
    }


# Print the results
def print_any_result(result):
    print("----- Verification Result -----")
    print("Best matched user :", result["matched_user"])
    print("Embedding score   :", round(result["best_embedding_score"], 4))
    print("MFCC score        :", round(result["best_mfcc_score"], 4))
    print("Final score       :", round(result["best_final_score"], 4))
    print("Accepted          :", result["accepted"])
    print("Bypass used       :", result["bypass_used"])
    print("State before      :", result["state_before"])
    print("State after       :", result["state_after"])

In [ ]:
# Example test file
TEST_FILE = os.path.join(ENROLLED_DIR, "Gaffar-near-1.m4a")

result = verify_any_user(TEST_FILE)
print_any_result(result)

----- Verification Result -----
Best matched user : Ziyad
Embedding score   : 0.9143
MFCC score        : 0.9995
Final score       : 0.9313
Accepted          : True
Bypass used       : False
State before      : Locked
State after       : Unlocked


In [ ]:
# Test PIN bypass
result_bypass = verify_any_user(TEST_FILE, pin="1234")
print_any_result(result_bypass)

----- Verification Result -----
Best matched user : None
Embedding score   : 0.0
MFCC score        : 0.0
Final score       : 0.0
Accepted          : True
Bypass used       : True
State before      : Locked
State after       : Unlocked


In [ ]:
# This checks each file against the full enrolled set
# and marks whether the best match belongs to the same speaker.
rows = []

for actual_speaker, files in speaker_files.items():
    for file_path in files:
        try:
            result = verify_any_user(file_path)

            rows.append({
                "actual_speaker": actual_speaker,
                "predicted_best_match": result["matched_user"],
                "file": os.path.basename(file_path),
                "embedding_score": result["best_embedding_score"],
                "mfcc_score": result["best_mfcc_score"],
                "final_score": result["best_final_score"],
                "accepted": result["accepted"],
                "correct_best_match": actual_speaker == result["matched_user"]
            })

        except Exception as e:
            print("Skipped:", file_path, "|", e)

df_results = pd.DataFrame(rows)
df_results.head()

,actual_speaker,predicted_best_match,file,embedding_score,mfcc_score,final_score,accepted,correct_best_match
0,Caroline,Caroline,Barriere-near-1.m4a,0.891217,0.999678,0.912909,False,True
1,Caroline,Caroline,Barriere-near-2.m4a,0.885935,0.999649,0.908678,False,True
2,Caroline,Caroline,Barriere-near-3.m4a,0.904143,0.999483,0.923211,True,True
3,Caroline,Caroline,Barriere-near-4.m4a,0.882638,0.998811,0.905872,False,True
4,Caroline,Caroline,Barriere-other-1.m4a,0.869302,0.998583,0.895158,False,True


In [ ]:
print("Acceptance rate:")
print(df_results["accepted"].mean())

print("\nBest-match correctness:")
print(df_results["correct_best_match"].mean())

print("\nAverage scores:")
print(df_results[["embedding_score", "mfcc_score", "final_score"]].mean())

Acceptance rate:
0.7840909090909091

Best-match correctness:
0.9886363636363636

Average scores:
embedding_score    0.930917
mfcc_score         0.999398
final_score        0.944613
dtype: float64


In [ ]:
# Example test file not from enrolled
NOT_ENROLLED_DIR = "/content/drive/MyDrive/project_dataset_audio/WakeWord_Dataset/other"
TEST_FILE = os.path.join(NOT_ENROLLED_DIR, "Ahmed-other-1.wav")

result = verify_any_user(TEST_FILE)
print_any_result(result)

----- Verification Result -----
Best matched user : Drishya
Embedding score   : 0.8016
MFCC score        : 0.9988
Final score       : 0.841
Accepted          : False
Bypass used       : False
State before      : Locked
State after       : Locked


In [ ]:
# Example test file not from enrolled
ENROLLED_DIR_2 = "/content/drive/MyDrive/project_dataset_audio"
TEST_FILE = os.path.join(ENROLLED_DIR_2, "Ziyad-test-1.m4a")

result = verify_any_user(TEST_FILE)
print_any_result(result)

----- Verification Result -----
Best matched user : Ziyad
Embedding score   : 0.9103
MFCC score        : 0.9984
Final score       : 0.9279
Accepted          : True
Bypass used       : False
State before      : Locked
State after       : Unlocked


WakeWord Detection

Using some of Activity 1 code

In [ ]:
# WAKE WORD PARAMETERS

WAKE_WORD_TEXT = "hey atlas"

# These should match the wake-word notebook settings
DURATION = 2.25
NUM_SAMPLES = int(TARGET_SR * DURATION)

# Threshold for wake-word classifier probability
WAKE_THRESHOLD = 0.70

In [ ]:
# Prepare input for wake-word CNN

# Pad / truncate audio to fixed duration for wake-word model
def pad_or_truncate_audio(y, num_samples=NUM_SAMPLES):
    if len(y) > num_samples:
        y = y[:num_samples]
    elif len(y) < num_samples:
        y = np.pad(y, (0, num_samples - len(y)))
    return y

def preprocess_for_wakeword(file_path):
    wav_path = convert_to_wav(file_path, target_sr=TARGET_SR)

    # Load audio
    y, sr = librosa.load(wav_path, sr=TARGET_SR, mono=True)

    # Force fixed duration
    y = pad_or_truncate_audio(y, NUM_SAMPLES)

    # Save temporary fixed-length wav so extract_mfcc can reuse it
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    temp_wav_path = tmp.name
    tmp.close()

    sf.write(temp_wav_path, y, sr)

    # Reuse your MFCC function from the notebook
    mfcc = extract_mfcc(temp_wav_path)

    # CNN expects shape like training data: (batch, n_mfcc, time, 1)
    mfcc = np.expand_dims(mfcc, axis=-1)
    mfcc = np.expand_dims(mfcc, axis=0)

    return mfcc

In [ ]:
# Detect wake word
def detect_wakeword(test_file, typed_wakeword=None, wake_model=None):
    state_before = "Sleep"

    # Bypass: typed wake word
    if typed_wakeword is not None and typed_wakeword.strip().lower() == WAKE_WORD_TEXT:
        return {
            "wake_probability": 1.0,
            "wake_detected": True,
            "state_before": state_before,
            "state_after": "Awake",
            "typed_bypass_used": True
        }

    if wake_model is None:
        raise ValueError("wake_model is None. Pass your trained wake-word model.")

    x_input = preprocess_for_wakeword(test_file)

    wake_prob = float(wake_model.predict(x_input, verbose=0)[0][0])
    wake_detected = wake_prob >= WAKE_THRESHOLD

    return {
        "wake_probability": wake_prob,
        "wake_detected": wake_detected,
        "state_before": state_before,
        "state_after": "Awake" if wake_detected else "Sleep",
        "typed_bypass_used": False
    }

# Print wake-word result
def print_wakeword_result(result):
    print("----- Wake Word Result -----")
    print("Wake probability :", round(result["wake_probability"], 4))
    print("Wake detected    :", result["wake_detected"])
    print("Typed bypass     :", result["typed_bypass_used"])
    print("State before     :", result["state_before"])
    print("State after      :", result["state_after"])

In [ ]:
def run_verification_wakeword_pipeline(
    verification_file,
    wakeword_file=None,
    verification_pin=None,
    typed_wakeword=None,
    wake_model=None
):
    assistant_state = {
        "lock_state": "Locked",
        "wake_state": "Sleep"
    }

    print("Initial state:", assistant_state)

    # Step 1: User verification
    verification_result = verify_any_user(verification_file, pin=verification_pin)

    print("\nUSER VERIFICATION")
    print_any_result(verification_result)

    if verification_result["accepted"]:
        assistant_state["lock_state"] = "Unlocked"
    else:
        assistant_state["lock_state"] = "Locked"
        assistant_state["wake_state"] = "Sleep"

        print("\nFinal state:", assistant_state)
        return {
            "verification_result": verification_result,
            "wakeword_result": None,
            "final_state": assistant_state
        }

    # At this point assistant is unlocked but still asleep
    assistant_state["wake_state"] = "Sleep"

    # --------------------------------------------------------
    # Step 2: Wake word detection
    # --------------------------------------------------------
    print("\nWAKE WORD DETECTION")

    if wakeword_file is None and typed_wakeword is None:
        print("No wake-word input provided. Remaining in Sleep mode.")
        print("\nFinal state:", assistant_state)
        return {
            "verification_result": verification_result,
            "wakeword_result": None,
            "final_state": assistant_state
        }

    wakeword_result = detect_wakeword(
        test_file=wakeword_file if wakeword_file is not None else verification_file,
        typed_wakeword=typed_wakeword,
        wake_model=wake_model
    )

    print_wakeword_result(wakeword_result)

    if wakeword_result["wake_detected"]:
        assistant_state["wake_state"] = "Awake"
    else:
        assistant_state["wake_state"] = "Sleep"

    print("\nFinal state:", assistant_state)

    return {
        "verification_result": verification_result,
        "wakeword_result": wakeword_result,
        "final_state": assistant_state
    }

##### Train the CNN Model

In [ ]:
PROCESSED_WAKEWORD_DIR = "/content/drive/MyDrive/project_dataset_audio/WakeWord_Dataset"

In [ ]:
for label in ["positive", "other", "near"]:
    folder = os.path.join(PROCESSED_WAKEWORD_DIR, label)
    print(f"\n{label.upper()}")
    for f in os.listdir(folder):
        print("  ", f)


POSITIVE
   Palmer-positive-1.wav
   Cella-positive-4.wav
   Shayne-positive-3.wav
   Khalaj-positive-3.wav
   Abderrahman-positive-2.wav
   Ostertag-positive-1.wav
   States-positive-3.wav
   Bowei-positive-4.wav
   Nair-positive-2.wav
   Wei-positive-2.wav
   Dahir-positive-3.wav
   Salman-positive-3.wav
   Palmer-positive-2.wav
   Rosenblatt-positive-4.wav
   Bowei-positive-1.wav
   Hanley-positive-3.wav
   Cherukunnummal Poothakandi-positive-3.wav
   KinanyAlaoui-positive-4.wav
   Udi-positive-3.wav
   Rosenblatt-positive-3.wav
   Ziping-positive-4.wav
   Salman-positive-2.wav
   Nandal-positive-3.wav
   Sharma-positive-1.wav
   Mourad-positive-4.wav
   liu-positive-2.wav
   Adjmal-positive-2.wav
   Junyang-positive-1.wav
   ElMalki-positive-2(1).wav
   Cheng-positive-3.wav
   Hamza-positive-3.wav
   Xiangrong-positive-3.wav
   Chang-positive-4.wav
   Cherukunnummal Poothakandi-positive-4.wav
   Arham-positive-1.wav
   Khoi-positive-4.wav
   Nandal-positive-4.wav
   Abderrahman-po

In [ ]:
# Build X and y for wake-word model
# X is the MFCCs (features), Y is the labels (output class)
# file_paths contains the path to the processed audio (so we can listen later to well/wrongly classified)

X_wake = []
y_wake = []
wake_file_paths = []

label_map = {
    "positive": 1,
    "other": 0,
    "near": 0
}

for label, class_id in label_map.items():
    folder = os.path.join(PROCESSED_WAKEWORD_DIR, label)

    for filename in os.listdir(folder):
        if not filename.endswith(".wav"):
            continue

        path = os.path.join(folder, filename)

        mfcc = extract_mfcc(path)

        X_wake.append(mfcc)
        y_wake.append(class_id)
        wake_file_paths.append(path)

X_wake = np.array(X_wake)
y_wake = np.array(y_wake)
wake_file_paths = np.array(wake_file_paths)

print("X_wake shape:", X_wake.shape)
print("y_wake shape:", y_wake.shape)

# add the channel dimension for CNN input
X_wake = X_wake[..., np.newaxis]

print("X_wake shape after channel dimension:", X_wake.shape)

X_wake shape: (603, 13, 91)
y_wake shape: (603,)
X_wake shape after channel dimension: (603, 13, 91, 1)


In [ ]:
X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X_wake,
    y_wake,
    wake_file_paths,
    test_size=0.10,
    random_state=42,
    stratify=y_wake
)

print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

Train samples: 542
Test samples: 61


In [ ]:
# CNN Model used to detect wake word
model = models.Sequential([
    layers.Input(shape=X_train.shape[1:]),

    # --- CNN BLOCK 1 ---
    layers.Conv2D(16, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # --- CNN BLOCK 2 ---
    layers.Conv2D(32, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # --- CNN BLOCK 3 ---
    layers.Conv2D(64, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2),

    # --- PREPARE FOR RNN ---
    layers.Reshape((-1, 64)),  # treat time axis as sequence

    # --- RNN LAYER ---
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.3),

    # --- DENSE HEAD ---
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])


# Complie model

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_w = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.6420 - loss: 0.6480 - val_accuracy: 0.6239 - val_loss: 0.6797
Epoch 2/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6744 - loss: 0.6325 - val_accuracy: 0.6239 - val_loss: 0.6622
Epoch 3/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.6628 - loss: 0.6578 - val_accuracy: 0.6239 - val_loss: 0.6664
Epoch 4/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.6674 - loss: 0.6484 - val_accuracy: 0.6239 - val_loss: 0.6652
Epoch 5/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6697 - loss: 0.6440 - val_accuracy: 0.6239 - val_loss: 0.6624
Epoch 6/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - accuracy: 0.6697 - loss: 0.6409 - val_accuracy: 0.6239 - val_loss: 0.6619
Epoch 7/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.6721 - loss: 0.6403 - val_accuracy: 0.6239 - val_loss: 0.6618
Epoch 8/100
109/109 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.6651 - loss: 0.6273 -

In [ ]:
loss_w, acc_w = model.evaluate(X_test, y_test, verbose=0)
print(f"Wake Model  Test Accuracy: {acc_w:.2f}")

Wake Model  Test Accuracy: 0.70


In [ ]:
EXPORT_DIR = "/content/drive/MyDrive/project_dataset_audio/saved_intent_model"
os.makedirs(EXPORT_DIR, exist_ok=True)


model.save_weights(os.path.join(EXPORT_DIR, "wake_model.weights.h5"))

##### Testing Verification + WakeWord

In [ ]:
WAKE_MODEL = model
# Example 1: Enrolled User + WakeWord Detected
verification_file = os.path.join(ENROLLED_DIR, "Gaffar-other-1.m4a")
wakeword_file = "/content/drive/MyDrive/project_dataset_audio/WakeWord_Dataset/positive/Gaffar-positive-1.wav"

pipeline_result = run_verification_wakeword_pipeline(
    verification_file=verification_file,
    wakeword_file=wakeword_file,
    wake_model=WAKE_MODEL
)

Initial state: {'lock_state': 'Locked', 'wake_state': 'Sleep'}

USER VERIFICATION
----- Verification Result -----
Best matched user : Ziyad
Embedding score   : 0.9202
MFCC score        : 0.9988
Final score       : 0.936
Accepted          : True
Bypass used       : False
State before      : Locked
State after       : Unlocked

WAKE WORD DETECTION
----- Wake Word Result -----
Wake probability : 0.9977
Wake detected    : True
Typed bypass     : False
State before     : Sleep
State after      : Awake

Final state: {'lock_state': 'Unlocked', 'wake_state': 'Awake'}


In [ ]:
# Example 2: Enrolled User + WakeWord not Detected
verification_file = os.path.join(ENROLLED_DIR, "Gaffar-other-1.m4a")
wakeword_file = "/content/drive/MyDrive/project_dataset_audio/WakeWord_Dataset/other/Salman-other-3.wav"

pipeline_result = run_verification_wakeword_pipeline(
    verification_file=verification_file,
    wakeword_file=wakeword_file,
    wake_model=WAKE_MODEL
)

Initial state: {'lock_state': 'Locked', 'wake_state': 'Sleep'}

USER VERIFICATION
----- Verification Result -----
Best matched user : Ziyad
Embedding score   : 0.9202
MFCC score        : 0.9988
Final score       : 0.936
Accepted          : True
Bypass used       : False
State before      : Locked
State after       : Unlocked

WAKE WORD DETECTION
----- Wake Word Result -----
Wake probability : 0.0
Wake detected    : False
Typed bypass     : False
State before     : Sleep
State after      : Sleep

Final state: {'lock_state': 'Unlocked', 'wake_state': 'Sleep'}


In [ ]:
# Example 3: Not Enrolled User + WakeWord Detected
verification_file = os.path.join(NOT_ENROLLED_DIR, "Ahmed-other-1.wav")
wakeword_file = "/content/drive/MyDrive/project_dataset_audio/WakeWord_Dataset/positive/Salman-positive-4.wav"

pipeline_result = run_verification_wakeword_pipeline(
    verification_file=verification_file,
    wakeword_file=wakeword_file,
    wake_model=WAKE_MODEL
)

Initial state: {'lock_state': 'Locked', 'wake_state': 'Sleep'}

USER VERIFICATION
----- Verification Result -----
Best matched user : Drishya
Embedding score   : 0.8016
MFCC score        : 0.9988
Final score       : 0.841
Accepted          : False
Bypass used       : False
State before      : Locked
State after       : Locked

Final state: {'lock_state': 'Locked', 'wake_state': 'Sleep'}


In [ ]:
# Example 4: Enrolled User + Near WakeWord
verification_file = os.path.join(ENROLLED_DIR, "Gaffar-other-1.m4a")
wakeword_file = "/content/drive/MyDrive/project_dataset_audio/WakeWord_Dataset/near/Salman-near-4.wav"

pipeline_result = run_verification_wakeword_pipeline(
    verification_file=verification_file,
    wakeword_file=wakeword_file,
    wake_model=WAKE_MODEL
)

Initial state: {'lock_state': 'Locked', 'wake_state': 'Sleep'}

USER VERIFICATION
----- Verification Result -----
Best matched user : Ziyad
Embedding score   : 0.9202
MFCC score        : 0.9988
Final score       : 0.936
Accepted          : True
Bypass used       : False
State before      : Locked
State after       : Unlocked

WAKE WORD DETECTION
----- Wake Word Result -----
Wake probability : 0.0781
Wake detected    : False
Typed bypass     : False
State before     : Sleep
State after      : Sleep

Final state: {'lock_state': 'Unlocked', 'wake_state': 'Sleep'}


In [ ]:
# Automatic Speech Recognition
# • Input: User’s voice.
# • Task: Transcribe the sentence spoken.
# • Output: Transcribed Speech shown on screen. Await confirmation.
# • By-pass: Allow the user to correct the transcription.

In [ ]:
# Install Whisper
!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 2.6 MB/s eta 0:00:00


In [ ]:
import whisper
asr_model = whisper.load_model("base")  # small, fast

100%|███████████████████████████████████████| 139M/139M [00:04<00:00, 33.1MiB/s]


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="whisper")

In [ ]:
!pip install sounddevice webrtcvad numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73520 sha256=1b6ec5114346c85f9ec708de783592dfe9f4801846838cbdc33ae53a11c1bfe7
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
Successfully built webrtcvad


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
ENROLLED_DIR = "/content/drive/MyDrive/project_dataset_audio/test_audio/raw"

Mounted at /content/drive


In [ ]:
import os

PROCESSED_DIR = "/content/drive/MyDrive/project_dataset_audio/test_audio/processed"
os.makedirs(os.path.join(PROCESSED_DIR), exist_ok=True)

In [ ]:
# For the conversion to wave files, we fix the Sampling rate and the duration
# Duration should be at least as long as the intent spoken duration

TARGET_SR = 16000    # 16 kHz
DURATION = 5         # seconds - covers most full-sentence intent commands
NUM_SAMPLES = int(TARGET_SR * DURATION)

In [ ]:
# Go through all files (.m4a format) and apply preprocessing


input_dir = ENROLLED_DIR
output_dir = PROCESSED_DIR

for filename in os.listdir(input_dir):     # for filename in os.listdir(input_dir)[:20]:
    if not filename.endswith(".m4a"):
        continue

    input_path = os.path.join(input_dir, filename)

    # Load + resample
    y, _ = librosa.load(input_path, sr=TARGET_SR, mono=True)

    # Pad or truncate
    if len(y) < NUM_SAMPLES:
        y = np.pad(y, (0, NUM_SAMPLES - len(y)))
    else:
        y = y[:NUM_SAMPLES]

    # Save as WAV
    output_filename = filename.replace(".m4a", ".wav")
    output_path = os.path.join(output_dir, output_filename)

    sf.write(output_path, y, TARGET_SR)

print("Conversion complete.")


/tmp/ipykernel_17080/2210553307.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(input_path, sr=TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_17080/2210553307.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(input_path, sr=TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_17080/2210553307.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(input_path, sr=TARGET_SR, mo

Conversion complete.


/tmp/ipykernel_17080/2210553307.py:14: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(input_path, sr=TARGET_SR, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
#Transcribing speech
WAV_FILE = os.path.join(output_dir, "Get_author.wav")
WAV_FILE2 = os.path.join(output_dir, "Openbook.wav")
WAV_FILE3 = os.path.join(output_dir, "Publish_year.wav")

WAV_FILE4 = os.path.join(output_dir, "Booksbyauthor.wav")
WAV_FILE5 = os.path.join(output_dir, "Goodbye.wav")
WAV_FILE6 = os.path.join(output_dir, "Dark_mode.wav")
WAV_FILE7 = os.path.join(output_dir, "Pottaotimer.wav")
WAV_FILE8 = os.path.join(output_dir, "Weathertoronto.wav")

WAV_FILE9 = os.path.join(output_dir, "Oos.wav")
speech = asr_model.transcribe(WAV_FILE) # Need change
speech2 = asr_model.transcribe(WAV_FILE2)
speech3 = asr_model.transcribe(WAV_FILE3)
speech4 = asr_model.transcribe(WAV_FILE4)

speech5 = asr_model.transcribe(WAV_FILE5)
speech6 = asr_model.transcribe(WAV_FILE6)
speech7 = asr_model.transcribe(WAV_FILE7)
speech8 = asr_model.transcribe(WAV_FILE8)

speech9 = asr_model.transcribe(WAV_FILE9)
#Transcribed Speech shown on screen. Await confirmation.
print(f"Detected speech is :{speech["text"]}")
print(f"Detected speech is :{speech2["text"]}")

print(f"Detected speech is :{speech3["text"]}")
print(f"Detected speech is :{speech4["text"]}")

print(f"Detected speech is :{speech5["text"]}")
print(f"Detected speech is :{speech6["text"]}")
print(f"Detected speech is :{speech7["text"]}")
print(f"Detected speech is :{speech8["text"]}")

print(f"Detected speech is :{speech9["text"]}")
#Bypass
print("Would you like to correct the transcription?")

#Fetch text to be passed to be next step


Detected speech is : Who wrote Twilight?
Detected speech is : Open Harry Potter Book
Detected speech is : When was Harry Potter published?
Detected speech is : Get books by order Leo Tolstoy
Detected speech is : Goodbye.
Detected speech is : Put in dark mode.
Detected speech is : Set Portator Timer for 15 minutes.
Detected speech is : What is the weather in Toronto?
Detected speech is : Please do my laundry.
Would you like to correct the transcription?


In [ ]:
#Intent Detection
# a) Out-Of-Scope (OOS): “Do my grocery shopping”
# b) Greetings: “Hello”
# c) Goodbye: “Thank you”
# d) Timer: “Set a soup timer for 2 minutes”
# e) Weather: “Does it rain in Paris today?”
# f) Book Domain Intent1- Get Book: "Get me the Dune book"
# g) Book Domain Intent2- Get Author: "Who authored Harry Potter"
# h) Book Domain Intent3- Get Publishing Year: "Which year was Twilight published"
# i) Book domain Intent4- Get Books by author: "List books written by Leo Tolstoy"

In [ ]:
set_timer_examples = [
("set a hot/B-ANAME potato/I-NAME timer for 5/B-DURATION minutes/I-DURATION"),
("set timer for 10/B-DURATION minutes/I-DURATION"),
("start a potato/B-ANAME timer for 30/B-DURATION seconds/I-DURATION"),
("start laundry/B-ANAME timer for 2/B-DURATION hours/I-DURATION"),
("set the kitchen/B-ANAME timer for 45/B-DURATION minutes/I-DURATION"),
("timer for 15/B-DURATION minutes/I-DURATION"),
("set the oven/B-ANAME timer for 1/B-DURATION hour/I-DURATION"),
("start an exam/B-ANAME timer for 20/B-DURATION minutes/I-DURATION"),
("please set a timer for 3/B-DURATION hours/I-DURATION"),
("set egg/B-ANAME timer for 90/B-DURATION seconds/I-DURATION"),
("set a timer for 25/B-DURATION minutes/I-DURATION"),
("timer for 40/B-DURATION minutes/I-DURATION"),
("start tomato/B-ANAME timer for 50/B-DURATION seconds/I-DURATION"),
("set a break/B-ANAME timer for 6/B-DURATION hours/I-DURATION"),
("please start a chess/B-ANAME timer for 12/B-DURATION minutes/I-DURATION"),
("could you set a break/B-ANAME timer for 13/B-DURATION minutes/I-DURATION"),
("set up a meditation/B-ANAME timer for 2/B-DURATION hours/I-DURATION please"),
("start a countdown for 45/B-DURATION seconds/I-DURATION"),
("begin pomodoro/B-ANAME timer for 28/B-DURATION minutes/I-DURATION"),
("i need a sleep/B-ANAME timer for 4/B-DURATION hours/I-DURATION"),
("activate a timer for 17/B-DURATION minutes/I-DURATION"),
("please start a countdown for 9/B-DURATION seconds/I-DURATION"),
("create a bomb/B-ANAME timer for 33/B-DURATION minutes/I-DURATION"),
("go ahead and set a nap/B-ANAME timer for 6/B-DURATION hours/I-DURATION"),
("launch a countdown/B-ANAME timer for 21/B-DURATION minutes/I-DURATION"),
("can you start a focus/B-ANAME timer for 3/B-DURATION hours/I-DURATION"),
("turn on a tea/B-ANAME timer for 55/B-DURATION seconds/I-DURATION"),
("i want a coffee/B-ANAME timer for 19/B-DURATION minutes/I-DURATION"),
("please activate timer for 10/B-DURATION hours/I-DURATION"),
("kick off a focus/B-ANAME timer for 24/B-DURATION minutes/I-DURATION")
]

In [ ]:
greeting_examples = [
    ("hello"),
    ("hi"),
    ("hey"),
    ("good morning"),
    ("good evening"),
    ("hello there"),
    ("hi assistant"),
    ("hey there"),
    ("good afternoon"),
    ("hello assistant"),
    ("hi there"),
    ("hey assistant"),
    ("good day"),
    ("hello again"),
    ("hi again"),
    ("greetings"),
    ("hiya"),
    ("hey buddy"),
    ("morning"),
    ("evening"),
    ("good to see you"),
    ("what’s up"),
    ("yo"),
    ("howdy"),
    ("nice to meet you"),
    ("pleased to meet you"),
    ("welcome back"),
    ("long time no see"),
    ("hey friend"),
    ("a very good morning to you"),

    ("hello friend"),
    ("hi friend"),
    ("hey mate"),
    ("hey pal"),
    ("sup"),
    ("what's going on"),
    ("what's new"),
    ("how's it going"),
    ("how are you"),
    ("how do you do"),
    ("nice to see you again"),
    ("good to see you again"),
    ("hey again"),
    ("hello hello"),
    ("hi hi"),
    ("hey hey"),
    ("morning there"),
    ("good morning to you"),
    ("good evening to you"),
    ("good afternoon to you"),
    ("welcome"),
    ("welcome friend"),
    ("glad to see you"),
    ("it's nice to see you"),
    ("yo there"),
    ("hey what's up"),
    ("hiya there"),
    ("hello buddy"),
    ("hello pal"),
    ("hey there friend"),
    ("good to have you here"),
    ("pleasure seeing you"),
    ("nice seeing you"),
    ("hello everyone"),
    ("hi everyone"),
    ("hey everyone"),
    ("good day to you"),
    ("what's up buddy"),
    ("how are things"),
    ("how have you been"),
    ("long time no talk"),
    ("it's been a while"),
    ("hello again friend"),
    ("hey stranger"),
    ("hi stranger"),
    ("greetings friend"),
    ("warm greetings"),
    ("salutations"),
    ("hello world")
]

In [ ]:
goodbye_examples = [
    ("bye"),
    ("goodbye"),
    ("see you"),
    ("talk to you later"),
    ("bye bye"),
    ("see you later"),
    ("good night"),
    ("catch you later"),
    ("farewell"),
    ("see you soon"),
    ("bye for now"),
    ("talk later"),
    ("have a good night"),
    ("goodbye assistant"),
    ("see you next time"),
    ("take care"),
    ("later"),
    ("see ya"),
    ("i’m heading out"),
    ("until next time"),
    ("have a great day"),
    ("have a good one"),
    ("peace out"),
    ("i’ll catch up with you later"),
    ("signing off"),
    ("that’s all for now"),
    ("have a nice evening"),
    ("talk again soon"),
    ("bye, see you around"),
    ("i’m off, goodbye"),
    ("I'm leaving now"),
    ("Have a nice day"),
    ("See you tomorrow"),
    ("Take it easy"),
    ("I have to go now"),
    ("Talk soon"),
    ("See you in a bit"),
    ("I'm signing out"),
    ("Until later"),
    ("See ya later"),
    ("I'm out"),
    ("Have a wonderful evening"),
    ("I'll be back later"),
    ("See you around"),
    ("Take care of yourself"),
    ("Catch you next time"),
    ("Bye for today"),
    ("I’ll talk to you later"),
    ("Have a pleasant day"),
    ("Time to go"),
    ("See you again"),
    ("I’m logging off"),
    ("Goodbye for now"),
    ("Enjoy the rest of your day"),
    ("Have a good evening"),
    ("I’ll see you later"),
    ("I’m heading off"),
    ("See you soon, bye"),
    ("Till next time"),
    ("Thanks, goodbye")
]


In [ ]:
oos_examples = [
("Take me to mueseum"),
("Tell me about yourself."),
("open my email"),
("tell me a joke"),
("who is the president"),
("translate hello to french"),
("turn on the lights"),
("book a flight"),
("what time is it"),
("search for restaurants"),
("play a movie"),
("open youtube"),
("increase volume"),
("show my calendar"),
("what is machine learning"),
("call mom"),
("let's dance"),
("Open Whatsapp"),
("Text dad"),
("Turn down the music"),
("Check inbox"),
("Launch spotify"),
("Reserve hotel room"),
("Show today's schedule"),
("Set a reminder for tomorrow"),
("Stream a horror film"),
("Add milk to shopping list"),
("Dim the bedroom lights"),
("play the latest news"),
("open netflix"),
("mute the sound"),
("show my reminders"),
("let's play a game"),
("open calendar app")
]

In [ ]:
Ask_for_weather_examples = [
    ("what is the weather in Ottawa/B-LOCATION today/B-DAY ?"),
    ("will it rain in Toronto/B-LOCATION tomorrow/B-DAY ?"),
    ("how is the weather in Montreal/B-LOCATION this/B-DAY weekend/I-DAY ?"),
    ("is it going to snow in Calgary/B-LOCATION tonight/B-DAY ?"),
    ("what’s the forecast for Vancouver/B-LOCATION next/B-DAY Monday/I-DAY ?"),
    ("tell me the weather in New/B-LOCATION York/I-LOCATION tomorrow/B-DAY"),
    ("will it be sunny in Paris/B-LOCATION this/B-DAY Sunday/I-DAY ?"),
    ("what is the temperature in Dubai/B-LOCATION today/B-DAY ?"),
    ("is it windy in Chicago/B-LOCATION next/B-DAY Tuesday/I-DAY ?"),
    ("how hot will it be in Miami/B-LOCATION tomorrow/B-DAY ?"),
    ("will it storm in London/B-LOCATION tonight/B-DAY ?"),
    ("what’s the weather like in Sydney/B-LOCATION this/B-DAY Friday/I-DAY ?"),
    ("is it cold in Berlin/B-LOCATION today/B-DAY ?"),
    ("what is tomorrow/B-DAY weather in Halifax/B-LOCATION ?"),
    ("will it snow in Edmonton/B-LOCATION next/B-DAY week/I-DAY ?"),
    ("what’s the forecast for Rome/B-LOCATION on/B-DAY Sunday/I-DAY ?"),
    ("is it raining in Tokyo/B-LOCATION right now today/B-DAY ?"),
    ("how’s the weather in Delhi/B-LOCATION tomorrow/B-DAY morning/I-DAY ?"),
    ("will it be humid in Bangkok/B-LOCATION this/B-DAY Saturday/I-DAY ?"),
    ("what is the high temperature in Madrid/B-LOCATION today/B-DAY ?"),
    ("is there a storm in Doha/B-LOCATION tomorrow/B-DAY ?"),
    ("what’s the weather in Amsterdam/B-LOCATION next/B-DAY Thursday/I-DAY ?"),
    ("will it be foggy in Seattle/B-LOCATION tonight/B-DAY ?"),
    ("tell me the forecast for Boston/B-LOCATION this/B-DAY weekend/I-DAY"),
    ("how windy is it in Riyadh/B-LOCATION today/B-DAY ?"),
    ("is it going to rain in Mumbai/B-LOCATION next/B-DAY Friday/I-DAY ?"),
    ("what’s the weather in Lisbon/B-LOCATION tomorrow/B-DAY afternoon/I-DAY ?"),
    ("will it freeze in Moscow/B-LOCATION tonight/B-DAY ?"),
    ("what’s the weather forecast in Zurich/B-LOCATION this/B-DAY Monday/I-DAY ?"),
    ("is it sunny in Barcelona/B-LOCATION tomorrow/B-DAY ?"),

    ("do I need an umbrella in Vancouver/B-LOCATION today/B-DAY ?"),
    ("should I expect rain in Dublin/B-LOCATION tomorrow/B-DAY ?"),
    ("what will the weather be like in Oslo/B-LOCATION this/B-DAY evening/I-DAY ?"),
    ("is it going to be hot in Cairo/B-LOCATION tomorrow/B-DAY ?"),
    ("what’s the temperature like in Los/B-LOCATION Angeles/I-LOCATION today/B-DAY ?"),
    ("will there be snow in Prague/B-LOCATION this/B-DAY weekend/I-DAY ?"),
    ("how is the climate in Reykjavik/B-LOCATION today/B-DAY ?"),
    ("what’s the weather update for Hong/B-LOCATION Kong/I-LOCATION tomorrow/B-DAY ?"),
    ("is it safe to travel in Bangkok/B-LOCATION weather today/B-DAY ?"),
    ("what’s the humidity in Singapore/B-LOCATION today/B-DAY ?"),
    ("will it rain heavily in Manila/B-LOCATION tonight/B-DAY ?"),
    ("is it sunny or cloudy in Rome/B-LOCATION tomorrow/B-DAY ?"),
    ("what’s the wind speed in Athens/B-LOCATION today/B-DAY ?"),
    ("how cold is it expected to be in Helsinki/B-LOCATION tonight/B-DAY ?"),
    ("will it be a nice day in San/B-LOCATION Francisco/I-LOCATION tomorrow/B-DAY ?"),
    ("what’s the UV index in Los/B-LOCATION Angeles/I-LOCATION today/B-DAY ?"),
    ("is it going to rain all day in London/B-LOCATION tomorrow/B-DAY ?"),
    ("what’s the weather outlook for Tokyo/B-LOCATION next/B-DAY week/I-DAY ?"),
    ("how warm will it get in Jakarta/B-LOCATION today/B-DAY ?"),
    ("will there be thunderstorms in Houston/B-LOCATION tonight/B-DAY ?"),
    ("what’s the forecast for Cape/B-LOCATION Town/I-LOCATION tomorrow/B-DAY ?"),
    ("is it freezing in Winnipeg/B-LOCATION today/B-DAY ?"),
    ("do I need a jacket in Stockholm/B-LOCATION tonight/B-DAY ?"),
    ("what’s the weather like outside in Buenos/B-LOCATION Aires/I-LOCATION today/B-DAY ?")
]

In [ ]:
get_book_examples = [
    ("Get the book Harry/B-BNAME Potter/I-BNAME"),
    ("Show the book The/B-BNAME Hunger/I-BNAME Games/O-BNAME"),
    ("Fetch book Twilight/B-BNAME"),
    ("Display book The/B-BNAME Hobbit/I-BNAME"),
    ("Get book Dune/B-BNAME"),
    ("Show me the book Pride/B-BNAME and/I-BNAME Prejudice/O-BNAME"),
    ("Fetch the book The/B-BNAME Great/I-BNAME Gatsby/O-BNAME"),
    ("Display book To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME"),
    ("Bring up book Brave/B-BNAME New/I-BNAME World/O-BNAME"),
    ("Retrieve book The/B-BNAME Catcher/I-BNAME in/I-BNAME the/I-BNAME Rye/O-BNAME"),
    ("Show the book Moby/B-BNAME Dick/O-BNAME"),
    ("Get me the book War/B-BNAME and/I-BNAME Peace/O-BNAME"),
    ("Fetch book The/B-BNAME Da/I-BNAME Vinci/I-BNAME Code/O-BNAME"),
    ("Display book The/B-BNAME Shining/O-BNAME"),
    ("Bring me the book The/B-BNAME Alchemist/O-BNAME"),
    ("Retrieve the book The/B-BNAME Kite/I-BNAME Runner/O-BNAME"),
    ("Show me book The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME"),
    ("Get the book The/B-BNAME Road/O-BNAME"),
    ("Fetch me book The/B-BNAME Name/I-BNAME of/I-BNAME the/I-BNAME Wind/O-BNAME"),
    ("Pull up book The/B-BNAME Lord/I-BNAME of/I-BNAME the/I-BNAME Rings/O-BNAME"),
    ("Find book The/B-BNAME Secret/I-BNAME Garden/O-BNAME"),
    ("Open book The/B-BNAME Old/I-BNAME Man/I-BNAME and/I-BNAME the/I-BNAME Sea/O-BNAME"),
    ("Load book The/B-BNAME Handmaid's/I-BNAME Tale/O-BNAME"),
    ("Show the book The/B-BNAME Picture/I-BNAME of/I-BNAME Dorian/I-BNAME Gray/O-BNAME"),
    ("Get me the book Crime/B-BNAME and/I-BNAME Punishment/O-BNAME"),
    ("Bring up the book The/B-BNAME Chronicles/I-BNAME of/I-BNAME Narnia/O-BNAME"),
    ("Fetch the book The/B-BNAME Book/I-BNAME Thief/O-BNAME"),
    ("Display the book The/B-BNAME Girl/I-BNAME with/I-BNAME the/I-BNAME Dragon/I-BNAME Tattoo/O-BNAME"),
    ("Retrieve book The/B-BNAME Road/I-BNAME"),
    ("Fetch me the book The/B-BNAME Alchemist/O-BNAME"),
    ("Pull up book The/B-BNAME Kite/I-BNAME Runner/O-BNAME"),
    ("Find book A/B-BNAME Tale/I-BNAME of/I-BNAME Two/I-BNAME Cities/O-BNAME"),
    ("Open book Brave/B-BNAME New/I-BNAME World/O-BNAME"),
    ("Load book To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME"),
    ("Show the book Wuthering/B-BNAME Heights/O-BNAME"),
    ("Get me the book Fahrenheit/B-BNAME 451/O-BNAME"),
    ("Bring up the book Jane/B-BNAME Eyre/O-BNAME"),
    ("Fetch the book Moby/B-BNAME Dick/I-BNAME"),
    ("Display the book War/B-BNAME and/I-BNAME Peace/O-BNAME"),
    ("Retrieve book The/B-BNAME Catcher/I-BNAME in/I-BNAME the/I-BNAME Rye/O-BNAME"),
    ("Fetch me the book Gone/B-BNAME with/I-BNAME the/I-BNAME Wind/O-BNAME"),
    ("Pull up the book The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME"),
    ("Find book The/B-BNAME Perks/I-BNAME of/I-BNAME Being/I-BNAME a/I-BNAME Wallflower/O-BNAME"),
    ("Open book The/B-BNAME Maze/I-BNAME Runner/O-BNAME"),
    ("Load book The/B-BNAME Da/I-BNAME Vinci/I-BNAME Code/O-BNAME"),
    ("Show the book The/B-BNAME Hunger/I-BNAME Games/O-BNAME"),
    ("Get me the book Twilight/B-BNAME"),
    ("Bring up the book Dune/B-BNAME"),
    ("Fetch the book The/B-BNAME Hobbit/O-BNAME"),
    ("Display book Pride/B-BNAME and/I-BNAME Prejudice/O-BNAME"),
    ("Retrieve book The/B-BNAME Shining/O-BNAME"),
    ("Fetch me book The/B-BNAME Great/I-BNAME Gatsby/O-BNAME"),
    ("Pull up book The/B-BNAME Chronicles/I-BNAME of/I-BNAME Narnia/O-BNAME"),
    ("Find book The/B-BNAME Book/I-BNAME Thief/O-BNAME"),
    ("Open book The/B-BNAME Girl/I-BNAME on/I-BNAME the/I-BNAME Train/O-BNAME"),
    ("Load book The/B-BNAME Night/I-BNAME Circus/O-BNAME"),
    ("Show the book Life/B-BNAME of/I-BNAME Pi/O-BNAME"),
    ("Get me the book The/B-BNAME Martian/O-BNAME"),
    ("Bring up the book Ready/B-BNAME Player/I-BNAME One/O-BNAME")

]

In [ ]:
get_author_examples = [
    ("Who wrote Twilight/B-BNAME ?"),
    ("Who authored Harry/B-BNAME Potter/I-BNAME ?"),
    ("Who wrote The/B-BNAME Hunger/I-BNAME Games/O-BNAME ?"),
    ("The/B-BNAME Thursday/I-BNAME Murder/I-BNAME Club/O-BNAME was written by whom ?"),
    ("Who wrote Dune/B-BNAME ?"),
    ("Who is the author of Pride/B-BNAME and/I-BNAME Prejudice/O-BNAME ?"),
    ("Who wrote The/B-BNAME Great/I-BNAME Gatsby/O-BNAME ?"),
    ("The/B-BNAME Catcher/I-INAME in/I-BNAME the/I-BNAME Rye/O-BNAME was written by whom ?"),
    ("Who authored To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME ?"),
    ("Who wrote Brave/B-BNAME New/I-BNAME World/O-BNAME ?"),
    ("Can you tell me who wrote The/B-BNAME Hobbit/O-BNAME ?"),
    ("Who is the writer of Moby/B-BNAME Dick/O-BNAME ?"),
    ("Who wrote War/B-BNAME and/I-BNAME Peace/O-BNAME ?"),
    ("The/B-BNAME Book/I-BNAME Thief/O-BNAME was written by whom ?"),
    ("Who authored The/B-BNAME Alchemist/O-BNAME ?"),
    ("Who wrote Crime/B-BNAME and/I-BNAME Punishment/O-BNAME ?"),
    ("Do you know who wrote The/B-BNAME Da/I-BNAME Vinci/I-BNAME Code/O-BNAME ?"),
    ("Who is the author of The/B-BNAME Shining/O-BNAME ?"),
    ("Who wrote The/B-BNAME Girl/I-BNAME with/I-BNAME the/I-BNAME Dragon/I-BNAME Tattoo/O-BNAME ?"),
    ("Who wrote The/B-BNAME Lord/I-BNAME of/I-BNAME the/I-BNAME Rings/O-BNAME ?"),
    ("Who is the author of The/B-BNAME Kite/I-BNAME Runner/O-BNAME ?"),
    ("Who authored The/B-BNAME Chronicles/I-BNAME of/I-BNAME Narnia/O-BNAME ?"),
    ("Can you tell me who wrote The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME ?"),
    ("The/B-BNAME Picture/I-BNAME of/I-BNAME Dorian/I-BNAME Gray/O-BNAME was written by whom ?"),
    ("Who wrote the bookThe/B-BNAME Road/O-BNAME ?"),
    ("Who is the writer of The/B-BNAME Handmaid's/I-BNAME Tale/O-BNAME ?"),
    ("Who authored The/B-BNAME Name/I-BNAME of/I-BNAME the/I-BNAME Wind/O-BNAME ?"),
    ("Do you know who wrote The/B-BNAME Secret/I-BNAME Garden/O-BNAME ?"),
    ("Who wrote The/B-BNAME Old/I-BNAME Man/I-BNAME and/I-BNAME the/I-BNAME Sea/O-BNAME ?"),

    ("Who is the author of 1984/B-BNAME ?"),
    ("Who wrote Animal/B-BNAME Farm/O-BNAME ?"),
    ("Who authored Fahrenheit/B-BNAME 451/O-BNAME ?"),
    ("Who wrote The/B-BNAME Catcher/I-BNAME in/I-BNAME the/I-BNAME Rye/O-BNAME ?"),
    ("Who is the writer of The/B-BNAME Odyssey/O-BNAME ?"),
    ("Who authored the book The/B-BNAME Iliad/O-BNAME ?"),
    ("Who wrote Les/B-BNAME Misérables/O-BNAME ?"),
    ("Who is the author of Dracula/B-BNAME ?"),
    ("Who wrote Frankenstein/B-BNAME ?"),
    ("Who authored the book Jane/B-BNAME Eyre/O-BNAME ?"),
    ("Who wrote Wuthering/B-BNAME Heights/O-BNAME ?"),
    ("Who is the writer of Don/B-BNAME Quixote/O-BNAME ?"),
    ("Who authored The/B-BNAME Brothers/I-BNAME Karamazov/O-BNAME ?"),
    ("Who wrote Lolita/B-BNAME ?"),
    ("Who is the author of The/B-BNAME Road/O-BNAME Less/I-BNAME Traveled/O-BNAME ?"),
    ("Who wrote Life/B-BNAME of/I-BNAME Pi/O-BNAME ?"),
    ("Who authored The/B-BNAME Martian/O-BNAME ?"),
    ("Who wrote Ready/B-BNAME Player/I-BNAME One/O-BNAME ?"),
    ("Who is the author of It/B-BNAME ?"),
    ("Who wrote the book Misery/B-BNAME ?"),
    ("Who authored Carrie/B-BNAME ?"),
    ("Who wrote the book The/B-BNAME Stand/O-BNAME ?"),
    ("Who is the writer of Pet/B-BNAME Sematary/O-BNAME ?"),
    ("Who authored The/B-BNAME Silmarillion/O-BNAME ?"),
    ("Who wrote Good/B-BNAME Omens/O-BNAME ?"),
    ("Who is the author of The/B-BNAME Book/I-BNAME Thief/O-BNAME ?"),
    ("Who wrote A/B-BNAME Game/I-BNAME of/I-BNAME Thrones/O-BNAME ?"),
    ("Who authored The/B-BNAME Winds/I-BNAME of/I-BNAME Winter/O-BNAME ?"),
    ("Who wrote The/B-BNAME Hobbit/O-BNAME series ?"),
    ("Who is the writer of the book Percy/B-BNAME Jackson/O-BNAME ?")
]

In [ ]:
get_publishing_year_examples = [

("What year was The/B-BNAME Hunger/I-BNAME Games/O-BNAME published ?"),
("When was Twilight/B-BNAME published ?"),
("What year was The/B-BNAME Book/I-BNAME Thief/O-BNAME published ?"),
("Which year was The/B-BNAME Lord/I-BNAME of/I-BNAME the/I-BNAME Rings/O-BNAME published ?"),
("Which year did The/B-BNAME Alchemist/O-BNAME come out ?"),
("The/B-BNAME Catcher/I-BNAME in/I-BNAME the/I-BNAME Rye/O-BNAME was published in which year ?"),
("What year was Dune/B-BNAME published ?"),
("When was Pride/B-BNAME and/B-INAME Prejudice/I-BNAME published ?"),
("Which year was The/B-BNAME Great/I-BNAME Gatsby/O-BNAME published ?"),
("What year did Moby/B-BNAME Dick/O-BNAME come out ?"),
("The/B-BNAME Hobbit/O-BNAME was published in which year ?"),
("When was To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME published ?"),
("Which year did Brave/B-BNAME New/I-BNAME World/O-BNAME come out ?"),
("What year was War/B-BNAME and/I-BNAME Peace/O-BNAME published ?"),
("When did The/B-BNAME Da/I-BNAME Vinci/I-BNAME Code/O-BNAME come out ?"),
("Which year was The/B-BNAME Shining/O-BNAME published ?"),
("The/B-BNAME Girl/I-BNAME with/I-BNAME the/I-BNAME Dragon/I-BNAME Tattoo/O-BNAME was published in which year ?"),
("What year was The/B-BNAME Kite/I-BNAME Runner/O-BNAME published ?"),
("When was The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME released ?"),
("Which year did The/B-BNAME Chronicles/I-BNAME of/I-BNAME Narnia/O-BNAME come out ?"),
("What year was The/B-BNAME Secret/I-BNAME Garden/O-BNAME published ?"),
("When was The/B-BNAME Old/I-BNAME Man/I-BNAME and/I-BNAME the/I-BNAME Sea/O-BNAME published ?"),
("Which year was The/B-BNAME Road/O-BNAME published ?"),
("What year did The/B-BNAME Handmaid's/I-BNAME Tale/O-BNAME come out ?"),
("When was The/B-BNAME Name/I-BNAME of/I-BNAME the/I-BNAME Wind/O-BNAME published ?"),
("The/B-BNAME Picture/I-BNAME of/I-BNAME Dorian/I-BNAME Gray/O-BNAME was published in which year ?"),
("What year was the book The/B-BNAME Hunger/I-BNAME Games/O-BNAME published ?"),
("When was the book Twilight/B-BNAME published ?"),
("What year was the book The/B-BNAME Book/I-BNAME Thief/O-BNAME published ?"),
("Which year was the book The/B-BNAME Lord/I-BNAME of/I-BNAME the/I-BNAME Rings/O-BNAME published ?"),
("Which year did the book The/B-BNAME Alchemist/O-BNAME come out ?"),
("The book The/B-BNAME Catcher/I-BNAME in/I-BNAME the/I-BNAME Rye/O-BNAME was published in which year ?"),
("What year was the book Dune/B-BNAME published ?"),
("When was the book Pride/B-BNAME and/I-BNAME Prejudice/O-BNAME published ?"),
("Which year was the book The/B-BNAME Great/I-BNAME Gatsby/O-BNAME published ?"),
("What year did the book Moby/B-BNAME Dick/O-BNAME come out ?"),
("The book The/B-BNAME Hobbit/O-BNAME was published in which year ?"),
("When was the book To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME published ?"),
("Which year did the book Brave/B-BNAME New/I-BNAME World/O-BNAME come out ?"),
("What year was the book War/B-BNAME and/I-BNAME Peace/O-BNAME published ?"),
("When did the book The/B-BNAME Da/I-BNAME Vinci/I-BNAME Code/O-BNAME come out ?"),
("Which year was the book The/B-BNAME Shining/O-BNAME published ?"),
("The book The/B-BNAME Girl/I-BNAME with/I-BNAME the/I-BNAME Dragon/I-BNAME Tattoo/O-BNAME was published in which year ?"),
("What year was the book The/B-BNAME Kite/I-BNAME Runner/O-BNAME published ?"),
("When was the book The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME released ?"),
("Which year did the book The/B-BNAME Chronicles/I-BNAME of/I-BNAME Narnia/O-BNAME come out ?"),
("What year was the book The/B-BNAME Secret/I-BNAME Garden/O-BNAME published ?"),
("When was the book The/B-BNAME Old/I-BNAME Man/I-BNAME and/I-BNAME the/I-BNAME Sea/O-BNAME published ?"),
("Which year was the book The/B-BNAME Road/O-BNAME published ?"),
("What year did the book The/B-BNAME Handmaid's/I-BNAME Tale/O-BNAME come out ?"),
("When was the book The/B-BNAME Name/I-BNAME of/I-BNAME the/I-BNAME Wind/O-BNAME published ?")
]

In [ ]:
get_books_by_author_examples = [
    ("Get books by J.K. Rowling/B-NAME"),
    ("Show me the books by Stephen/B-NAME King/I-NAME)"),
    ("Fetch books by George/B-NAME Orwell/I-NAME"),
    ("Display books by Charles/B-NAME Dickens/I-NAME"),
    ("Get books by Ernest/B-NAME Hemingway/I-NAME"),
    ("Show me the books written by J.R.R. Tolkien/B-NAME"),
    ("Fetch books authored by F. Scott Fitzgerald/B-NAME"),
    ("Display books by Harper/B-NAME Lee/I-NAME"),
    ("Get books by Jane/B-NAME Austen/I-NAME"),
    ("Show me the books by Mark/B-NAME Twain/I-NAME"),
    ("Fetch books by Leo/B-NAME Tolstoy/I-NAME"),
    ("Display books by Fyodor/B-NAME Dostoevsky/I-NAME"),
    ("Get books by Agatha/B-NAME Christie/I-NAME"),
    ("Show me the books written by author Dan/B-NAME Brown/I-NAME"),
    ("Fetch books authored by Paulo/B-NAME Coelho/I-NAME"),
    ("Display books authored by J.D./B-NAME Salinger/I-NAME"),
    ("Get books by Virginia/B-NAME Woolf/I-NAME"),
    ("Show me the books by Oscar/B-NAME Wilde/I-NAME"),
    ("Fetch books by writer Emily/B-NAME Brontë/I-NAME"),
    ("Display books by author Herman/B-NAME Melville/I-NAME"),
    ("Get books by Arthur/B-NAME Conan/I-NAME Doyle/O-NAME"),
    ("Show me the books authored by Stephanie/B-NAME Meyer/I-NAME"),
    ("Show me the books written by Kurt/B-NAME Vonnegut/I-NAME"),
    ("Show me the books by Kurt/B-NAME Vonnegut/I-NAME"),
    ("Fetch books by Gabriel/B-NAME García/I-NAME Márquez/O-NAME"),
    ("Get books by Salman/B-NAME Rushdie/I-NAME"),
    ("Show me the books by Kazuo/B-NAME Ishiguro/I-NAME"),
    ("Fetch books by Isabel/B-NAME Allende/I-NAME"),
    ("Display books by Margaret/B-NAME Atwood/I-NAME"),
    ("Get books by Neil/B-NAME Gaiman/I-NAME"),
    ("Show me the books written by writer Chimamanda/B-NAME Ngozi/I-NAME Adichie/O-NAME"),
    ("Fetch books authored by Haruki/B-NAME Murakami/I-NAME"),
    ("Display books by C.S./B-NAME Lewis/I-NAME"),
    ("Get books by Roald/B-NAME Dahl/I-NAME"),
    ("Show me the books by John/B-NAME Steinbeck/I-NAME"),
    ("Fetch books by Toni/B-NAME Morrison/I-NAME"),
    ("Display books by James/B-NAME Joyce/I-NAME"),
    ("Get books by author Vladimir/B-NAME Nabokov/I-NAME"),
    ("Show me the books written by Ian/B-NAME McEwan/I-NAME"),
    ("Fetch books authored by Philip/B-NAME Roth/I-NAME"),
    ("Display books by Zadie/B-NAME Smith/I-NAME"),
    ("Get books by Jhumpa/B-NAME Lahiri/I-NAME"),
    ("Show me the books by Colson/B-NAME Whitehead/I-NAME"),
    ("Fetch books by Donna/B-NAME Tartt/I-NAME"),
    ("Display books by Anthony/B-NAME Doerr/I-NAME"),
    ("Get books by George/B-NAME R.R./I-NAME Martin/O-NAME"),
    ("Show me the books by Patrick/B-NAME Rothfuss/I-NAME"),
    ("Fetch books by Brandon/B-NAME Sanderson/I-NAME"),
    ("Display books by Suzanne/B-NAME Collins/I-NAME"),
    ("Get books by Veronica/B-NAME Roth/I-NAME"),
    ("Show me the books by Rick/B-NAME Riordan/I-NAME"),
    ("Fetch books by Stephenie/B-NAME Meyer/I-NAME"),
    ("Display books by E.L./B-NAME James/I-NAME"),
    ("Get books written by Andy/B-NAME Weir/I-NAME"),
    ("Show me the books by Madeline/B-NAME Miller/I-NAME")
]

In [ ]:
dark_mode_examples = [
    ("Enable dark mode"),
    ("Switch to dark mode"),
    ("Turn on dark mode"),
    ("Activate dark mode"),
    ("Use dark theme"),
    ("Set theme to dark"),
    ("Change to dark theme"),
    ("Make the interface dark"),
    ("Apply dark mode"),
    ("Go to dark mode"),
    ("I want dark mode"),
    ("Can you enable dark mode"),
    ("Dark mode please"),
    ("Switch the theme to dark"),
    ("Turn on the night mode"),
    ("Use the dark color scheme"),
    ("Set display to dark"),
    ("Change color theme to dark"),
    ("Switch to night mode"),
    ("Enable night theme"),
    ("Turn on night mode"),
    ("Activate night theme"),
    ("Use the night mode"),
    ("Set theme to night"),
    ("Change to night theme"),
    ("Enable the dark theme"),
    ("Switch on dark mode"),
    ("Turn the theme dark"),
    ("Make everything dark"),
    ("Set dark mode on"),
    ("Apply the dark theme"),
    ("Go dark"),
    ("Use dark display mode"),
    ("Change the interface to dark"),
    ("Set the UI to dark"),
    ("Switch the interface to dark mode"),
    ("Activate the dark color scheme"),
    ("Enable dark display"),
    ("Turn on dark appearance"),
    ("Set appearance to dark"),
    ("Change appearance to dark mode"),
    ("Make the screen dark themed"),
    ("Switch over to dark theme"),
    ("Use dark styling"),
    ("Dark theme please"),
    ("Please enable dark mode"),
    ("Please switch to dark mode"),
    ("Could you turn on dark mode"),
    ("Can you switch to dark theme"),
    ("I prefer dark mode"),
    ("Give me dark mode"),
    ("Put it in dark mode"),
    ("Make it night mode"),
    ("Enable nighttime mode"),
    ("Turn everything to dark mode")
]

In [ ]:
open_book_examples = [
    ("Open book Harry/B-BNAME Potter/I-BNAME"),
    ("Open book The/B-BNAME Hunger/I-BNAME Games/O-BNAME"),
    ("Open book Twilight/B-BNAME"),
    ("Open the book The/B-BNAME Hobbit/O-BNAME"),
    ("Open book Dune/B-BNAME"),
    ("Open book Pride/B-BNAME and/B-BNAME Prejudice/I-BNAME"),
    ("Open the book The/B-BNAME Great/I-BNAME Gatsby/O-BNAME"),
    ("Open book The/B-BNAME Catcher/I-BNAME in/I-BNAME the/I-INAME Rye/O-BNAME"),
    ("Open book To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME"),
    ("Open book Brave/B-BNAME New/I-BNAME World/O-BNAME"),
    ("Open the book The/B-BNAME Da/I-BNAME Vinci/I-BNAME Code/O-BNAME"),
    ("Open book The/B-BNAME Shining/I-BNAME"),
    ("Open book The/B-BNAME Lord/I-BNAME of/I-BNAME the/I-BNAME Rings/O-BNAME"),
    ("Open book The/B-BNAME Alchemist/O-BNAME"),
    ("Open the book Moby/B-BNAME Dick/I-BNAME"),
    ("Open book War/B-BNAME and/I-BNAME Peace/O-BNAME"),
    ("Open book The/B-BNAME Chronicles/I-BNAME of/I-BNAME Narnia/O-BNAME"),
    ("Open the book Crime/B-BNAME and/I-BNAME Punishment/O-BNAME"),
    ("Open book The/B-BNAME Kite/I-BNAME Runner/O-BNAME"),
    ("Open book Jane/B-BNAME Eyre/O-BNAME"),
    ("Open the book Wuthering/B-BNAME Heights/O-BNAME"),
    ("Open book The/B-BNAME Book/I-BNAME Thief/O-BNAME"),
    ("Open book Fahrenheit/B-BNAME 451/O-BNAME"),
    ("Open the book Animal/B-BNAME Farm/O-BNAME"),
    ("Open book The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME"),
    ("Open book A/B-BNAME Tale/I-BNAME of/I-BNAME Two/I-BNAME Cities/O-BNAME"),
    ("Open the book The/B-BNAME Girl/I-BNAME with/I-BNAME the/I-BNAME Dragon/I-BNAME Tattoo/O-BNAME"),
    ("Open book The/B-BNAME Picture/I-BNAME of/I-BNAME Dorian/I-BNAME Gray/O-BNAME"),
    ("Open book Les/B-BNAME Misérables/I-BNAME"),
    ("Open the book The/B-BNAME Perks/I-BNAME of/I-BNAME Being/I-BNAME a/I-BNAME Wallflower/O-BNAME"),
    ("Open book Gone/B-BNAME with/I-BNAME the/I-BNAME Wind/O-BNAME"),
    ("Open the book The/B-BNAME Maze/I-BNAME Runner/O-BNAME"),
    ("Open the book The/B-BNAME Fault/I-BNAME in/I-BNAME Our/I-BNAME Stars/O-BNAME"),
    ("Open the book A/B-BNAME Tale/I-BNAME of/I-BNAME Two/I-BNAME Cities/O-BNAME"),
    ("Open the book To/B-BNAME Kill/I-BNAME a/I-BNAME Mockingbird/O-BNAME"),
    ("Open the book Brave/B-BNAME New/I-BNAME World/O-BNAME"),
]

In [ ]:
next_page_examples = [
    ("Flip to next page"),
    ("Go to next page"),
    ("Next page"),
    ("Next page please"),
    ("Move to next page"),
    ("Take me to the next page"),
    ("Show next page"),
    ("Load next page"),
    ("Advance to next page"),
    ("Go forward a page"),
    ("Page forward"),
    ("Scroll to next page"),
    ("Jump to next page"),
    ("Navigate to next page"),
    ("Continue to next page"),
    ("Show me the next page"),
    ("Can you go to the next page"),
    ("Switch to the next page"),
    ("Forward page"),
    ("Next page please"),
    ("Move forward a page"),
    ("Get next page please"),
    ("Get to the next page"),
    ("Take me forward a page"),
    ("Head to the next page"),
    ("Skip to next page"),
    ("Proceed to next page"),
    ("Page ahead"),
    ("Go one page forward"),
    ("Bring up the next page"),
    ("Go to the following page"),
    ("Open the next page"),
    ("Turn to the next page"),
    ("Flip forward a page"),
    ("Advance one page"),
    ("Move ahead a page"),
    ("Step to the next page"),
    ("Shift to the next page"),
    ("Take me ahead one page"),
    ("Show the following page"),
    ("Load the following page"),
    ("Continue forward a page"),
    ("Proceed forward"),
    ("Go ahead one page"),
    ("Scroll forward a page"),
    ("Jump ahead a page"),
    ("Navigate forward a page"),
    ("Forward to the next one"),
    ("Next page now"),
    ("Let’s see the next page"),
    ("Move to the following page"),
    ("Bring me to the next page"),
    ("Go forward to the next"),
    ("Skip ahead one page"),
    ("Advance to the following page"),
    ("Head forward a page"),
    ("Push to the next page"),
    ("On to the next page"),
    ("Take it to the next page"),
    ("Move along to the next page")
]

In [ ]:
increase_font_examples = [
    ("Increase font size"),
    ("Zoom in"),
    ("Increase font"),
    ("Zoom in please"),
    ("Zoom in a bit"),
    ("Make the text bigger"),
    ("Bump up the font size"),
    ("Make font larger"),
    ("Scale up the text"),
    ("Enlarge the text"),
    ("Make it bigger"),
    ("Bigger text please"),
    ("Can you zoom in"),
    ("Zoom in a little"),
    ("Increase text size"),
    ("Make the letters bigger"),
    ("Boost the font size"),
    ("Zoom in more"),
    ("Zoom in slightly"),
    ("Zoom in further"),
    ("Zoom in just a bit"),
    ("Zoom in a little more"),
    ("Make it a bit bigger"),
    ("Make it slightly bigger"),
    ("Make the text a bit larger"),
    ("Increase the font size a bit"),
    ("Increase text size a little"),
    ("Can you make it bigger"),
    ("Can you enlarge the text"),
    ("Can you increase the font"),
    ("Could you zoom in"),
    ("Could you make the text bigger"),
    ("Could you increase text size"),
    ("Please zoom in"),
    ("Please make the text bigger"),
    ("Please enlarge the font"),
    ("Please increase font size"),
    ("I need bigger text"),
    ("I want the text larger"),
    ("The text is too small, make it bigger"),
    ("Make everything bigger"),
    ("Scale the text up"),
    ("Turn up the font size"),
    ("Adjust the font to be larger"),
    ("Expand the text size"),
    ("Raise the text size"),
    ("Up the font size")
]


In [ ]:
decrease_font_examples = [
    ("Decrease font size"),
    ("Zoom out"),
    ("Decrease font"),
    ("Zoom out please"),
    ("Make the text smaller"),
    ("Make the text smaller"),
    ("Reduce the font size"),
    ("Make font smaller"),
    ("Scale down the text"),
    ("Shrink the text"),
    ("Make it smaller"),
    ("Smaller text please"),
    ("Can you zoom out"),
    ("Zoom out a little"),
    ("Decrease text size"),
    ("Make the letters smaller"),
    ("Lower the font size"),
    ("Reduce text size"),
    ("Shrink the font"),
    ("Zoom out a bit"),
    ("Make the font tinier"),
    ("Scale back the text"),
    ("Bring down the font size"),
    ("Can you shrink the text"),
    ("Decrease the text size please"),
    ("Reduce the font"),
    ("Lower text size"),
    ("Make font size  a bit smaller"),
    ("Make font size slightly smaller"),
    ("Shrink font size down"),
    ("Decrease text size a little"),
    ("Turn down the font size"),
    ("Drop the text size"),
    ("Set font size lower"),
    ("Adjust the text smaller"),
    ("Make the display text smaller"),
    ("Can you make text size smaller"),
    ("Could you shrink the text"),
    ("Please decrease the font size"),
    ("Please make the text smaller"),
    ("Please reduce the text size"),
    ("I need smaller text"),
    ("It’s too big, make it smaller"),
    ("The text is too large"),
    ("Bring the size down"),
    ("Scale the font down"),
    ("Reduce font size a bit"),
    ("Make things smaller"),
    ("Cut down the text size"),
    ("Shrink it a little more"),
    ("Minimize the font size"),
    ("Tone down the text size"),
    ("Dial down the font"),
    ("Compress the text size")
]

In [ ]:
increase_brightness_examples = [
    ("Increase brightness"),
    ("Brighten"),
    ("Increase brightness please"),
    ("Brighten please"),
    ("Raise the brightness"),
    ("Boost the brightness"),
    ("Turn up the brightness"),
    ("Make it brighter"),
    ("Brighten the screen"),
    ("Brighten it a bit"),
    ("Brighten it up"),
    ("Increase the screen brightness"),
    ("Make the screen brighter"),
    ("Turn the brightness up"),
    ("Bump up the brightness"),
    ("Raise screen brightness"),
    ("Make it more bright"),
    ("Can you brighten the screen"),
    ("Brighten it please"),
    ("Bring the brightness up"),
    ("Scale up the brightness"),
    ("Lighten the screen"),
    ("Max out the brightness"),
    ("Up the brightness please"),
    ("Please increase brightness"),
    ("Turn up the screen brightness"),
    ("Make it a bit brighter"),
    ("Make it slightly brighter"),
    ("Boost the brightness"),
    ("Raise the brightness level"),
    ("Crank up the brightness"),
    ("Set brightness higher"),
    ("Adjust brightness up"),
    ("Brighten the display"),
    ("Make the display brighter"),
    ("Add more brightness"),
    ("Can you increase the brightness"),
    ("Could you brighten it"),
    ("Please increase the brightness"),
    ("Please make it brighter"),
    ("I need it brighter"),
    ("It’s too dark, brighten it"),
    ("The screen is too dim"),
    ("Give it more light"),
    ("Turn it up a notch"),
    ("Increase the light level"),
    ("Make things brighter"),
    ("Light it up a bit"),
    ("Bring more light to the screen"),
    ("Push the brightness higher"),
    ("Enhance the brightness"),
    ("Dial up the brightness"),
    ("Turn the lights up"),
    ("Lift the brightness a bit")
]

In [ ]:
decrease_brightness_examples = [
    ("Decrease brightness"),
    ("Dim"),
    ("Decrease brightness please"),
    ("Dim please"),
    ("Lower the brightness"),
    ("Reduce brightness"),
    ("Turn down the brightness"),
    ("Make it dimmer"),
    ("Dim the screen"),
    ("Dim it a bit"),
    ("Dim it down"),
    ("Tone down the brightness"),
    ("Make the screen darker"),
    ("Decrease the screen brightness"),
    ("Lower screen brightness"),
    ("Turn the brightness down"),
    ("Reduce the screen light"),
    ("Dim the display"),
    ("Make it less bright"),
    ("Can you dim the screen"),
    ("Dim it please"),
    ("Bring the brightness down"),
    ("Scale down the brightness"),
    ("Darken the screen"),
    ("Lower the brightness"),
    ("Reduce brightness"),
    ("Make it a little dimmer"),
    ("Make it slightly darker"),
    ("Decrease brightness a bit"),
    ("Turn the screen brightness down"),
    ("Drop the brightness"),
    ("Bring the screen light down"),
    ("Cut down the brightness"),
    ("Ease off the brightness"),
    ("Set brightness lower"),
    ("Adjust brightness down"),
    ("Dim the screen a little"),
    ("Dim the display a bit more"),
    ("Make the display less bright"),
    ("Can you lower the brightness"),
    ("Could you dim it"),
    ("Please lower the brightness"),
    ("Please make it dimmer"),
    ("Please reduce brightness"),
    ("I need it darker"),
    ("It’s too bright, dim it"),
    ("The screen is too bright"),
    ("Reduce the light a bit"),
    ("Make things darker"),
    ("Lower the light level"),
    ("Turn light down a notch"),
    ("Soften the brightness"),
    ("Decrease the light intensity"),
    ("Bring it down a bit")
]

In [ ]:
# With the inline annotation, we have the slot names, and the other tokens would be assigned "O"
# outside (in the BIO annotation schema)

def parse_example(sentence):
    tokens = []
    slots = []

    for word in sentence.split():

        if "/" in word:
            token, slot = word.rsplit("/", 1)
        else:
            token = word
            slot = "O"

        tokens.append(token)
        slots.append(slot)

    return tokens, slots

In [ ]:
# We gather all tokens, slots and intents
# For each sentence we have as many slots as tokens, but a single intent

all_tokens = []
all_slots = []
all_intents = []

intent_map = {
    "Greeting": greeting_examples,
    "Goodbye": goodbye_examples,
    "OOS": oos_examples,
    "AskForWeather": Ask_for_weather_examples,
    "GetBook": get_book_examples,
    "GetAuthor": get_author_examples,
    "GetPublishingYear": get_publishing_year_examples,
    "GetBooksByAuthor": get_books_by_author_examples,
    "EnableDarkMode": dark_mode_examples,
    "OpenBook": open_book_examples,
    "NextPage": next_page_examples,
    "IncreaseFont": increase_font_examples,
    "DecreaseFont": decrease_font_examples,
    "IncreaseBrightness": increase_brightness_examples,
    "DecreaseBrightness": decrease_brightness_examples,
    "SetTimer": set_timer_examples
}

for intent_name, sentences in intent_map.items():
    for s in sentences:
        # if s is a tuple like (sentence_str, intent), unpack it
        if isinstance(s, tuple):
            sentence_text = s[0]
        else:
            sentence_text = s

        tokens, slots = parse_example(sentence_text)
        all_tokens.append(tokens)
        all_slots.append(slots)
        all_intents.append(intent_name)

In [ ]:
#Creating label dictionaries for model to learn

In [ ]:
# gathers the set of slots (no need to predefine them)
unique_slots = set()

for seq in all_slots:
    unique_slots.update(seq)

unique_slots = sorted(list(unique_slots))
unique_slots

['B-ANAME',
 'B-BNAME',
 'B-DAY',
 'B-DURATION',
 'B-INAME',
 'B-LOCATION',
 'B-NAME',
 'I-BNAME',
 'I-DAY',
 'I-DURATION',
 'I-INAME',
 'I-LOCATION',
 'I-NAME',
 'I-NAME)',
 'O',
 'O-BNAME',
 'O-NAME']

In [ ]:
# creates the mapping
slot2id = {s:i for i,s in enumerate(unique_slots)}
id2slot = {i:s for s,i in slot2id.items()}

In [ ]:
# gather the set of intents (even if those were predefined)
unique_intents = sorted(list(set(all_intents)))
unique_intents

['AskForWeather',
 'DecreaseBrightness',
 'DecreaseFont',
 'EnableDarkMode',
 'GetAuthor',
 'GetBook',
 'GetBooksByAuthor',
 'GetPublishingYear',
 'Goodbye',
 'Greeting',
 'IncreaseBrightness',
 'IncreaseFont',
 'NextPage',
 'OOS',
 'OpenBook',
 'SetTimer']

In [ ]:
# creates the mapping
intent2id = {s:i for i,s in enumerate(unique_intents)}
id2intent = {i:s for s,i in intent2id.items()}

In [ ]:
#Intent Alignment + Slot Alignment + Padding

In [ ]:
# A vector with all intent identifiers is generated
# It is of size  (with all the samples) - would change with changing the size of the dataset

intent_labels_ids = [intent2id[i] for i in all_intents]

In [ ]:
# The AutoTokenizer figures out which Tokenizer is required for the model
# Here it is the Word Piece tokenizer

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Exploring tokenizer
# We set the maximum sentence length to 10, and including padding to max_length
# We include truncation if sentence is more than max_length
# Additional tokens are included for BERT, the [CLS] - 101, and the [SEP] - 102

sentence = "Set a nameless alarm for 7 am on Sunday"
encoding = tokenizer(sentence, padding="max_length", truncation = True, max_length = 10, return_tensors="pt")
print(encoding)

{'input_ids': tensor([[ 101, 2275, 1037, 2171, 3238, 8598, 2005, 1021, 2572,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [ ]:
# We notice that the sentence above (set a nameless alarm for 7 am) has 7 tokens
# but the resulting encoding had 10 tokens with the [CLS] - 101, and the [SEP] - 102
# Let's see the effect of Word piece tokenization

input_ids = encoding["input_ids"][0]  # tensor of token IDs
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Input IDs:", input_ids)
print("Tokens:", tokens)

Input IDs: tensor([ 101, 2275, 1037, 2171, 3238, 8598, 2005, 1021, 2572,  102])
Tokens: ['[CLS]', 'set', 'a', 'name', '##less', 'alarm', 'for', '7', 'am', '[SEP]']


In [ ]:
# We do the encoding for all the sentences (all_tokens)
# The padding=True will actually use the longest sentence as the maximum size

encodings = tokenizer(
    all_tokens,
    is_split_into_words=True,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

In [ ]:
# Align at the word piece level
# Takes all_tokens (from the original sentences) and their slots
# Takes the result of the word piece tokenizer (endodings) and the set of indices for the slots

# Example with
# Original tokens ["Set", "a", "nameless", "alarm", "for", "7", "am"]
# Slots ["O", "O", "O", "O", "O", "B-TIME", "I-TIME"]
# Word Piece tokens ['[CLS]', 'set', 'a', 'name', '##less', 'alarm', 'for', '7', 'am', '[SEP]']
# Word IDs: [None, 0, 1, 2, 2, 3, 4, 5, 6, None]
# Aligned labels should be: [-100, O, O, O, -100, O, O, B-TIME, I-TIME, -100]
# Everything labeled -100 will not be considered in the learning process

def align_labels(all_tokens, all_slots, encodings, slot2id):

    aligned_labels = []

    for i in range(len(all_tokens)):  # go through each sentence (i is the sentence index)

        word_ids = encodings.word_ids(batch_index=i)  # word_ids actually points to the word index in the original sentence
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:   # go through the words in the sentence

            if word_id is None:
                label_ids.append(-100)

            elif word_id != previous_word_id:
                label_ids.append(slot2id[all_slots[i][word_id]])

            else:
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    return aligned_labels

In [ ]:
# Call the previously defined method
aligned_slot_labels = align_labels(
    all_tokens,
    all_slots,
    encodings,
    slot2id
)

In [ ]:
#Train/Test Split + PyTorch Dataset
# We prepare for training by separating into train/test (more correctly called validation - val)
# We use 20% for test

from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    list(range(len(intent_labels_ids))),  # The set of sentences
    test_size=0.2,
    random_state=42
)

def subset(lst, indices):
    return [lst[i] for i in indices]

train_encodings_split = {
    "input_ids": encodings["input_ids"][train_idx],
    "attention_mask": encodings["attention_mask"][train_idx],
}

val_encodings_split = {
    "input_ids": encodings["input_ids"][val_idx],
    "attention_mask": encodings["attention_mask"][val_idx],
}

# For the slot filling train / validation
train_slots = subset(aligned_slot_labels, train_idx)
val_slots   = subset(aligned_slot_labels, val_idx)

# For the intent classification train / validation
train_intents = subset(intent_labels_ids, train_idx)
val_intents   = subset(intent_labels_ids, val_idx)

In [ ]:
# We need to transform our training set into a PyTorch Dataset
# as we will use a PyTorch model for the learning, so it needs to be in the required format
# The class contains a getitem to access a single example from the dataset

import torch
from torch.utils.data import Dataset

class JointDataset(Dataset):
    def __init__(self, encodings, slot_labels, intent_labels):
        self.encodings = encodings
        self.slot_labels = slot_labels
        self.intent_labels = intent_labels

    def __getitem__(self, idx):
        item = {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "slot_labels": torch.tensor(self.slot_labels[idx]),
            "intent_label": torch.tensor(self.intent_labels[idx]),
        }
        return item

    def __len__(self):
        return len(self.intent_labels)

In [ ]:
# The actual transformation into a PyTorch Dataset (format required for the training and validation)

# Training set in proper format
train_dataset = JointDataset(
    train_encodings_split,
    train_slots,
    train_intents
)

# Validation set in proper format
val_dataset = JointDataset(
    val_encodings_split,
    val_slots,
    val_intents
)

In [ ]:
#Joint Intent + Slot Model
# The AutoModel knows about the last layer of each model (here an encoder - DistilBERT)
# Here the DistilBert provides hidden states and we will need to add a classifier on top

from transformers import AutoModel

model_name = "distilbert-base-uncased"
encoder = AutoModel.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Here is where we specifically provide the architecture to do transfer learning from BERT to our classification tasks

# We require a classifier on top of the [CLS] output for the intent
# (5 classes - numIntents: set_timer, set_alarm, oos, greeting, goodbye)

# We require a classifier on top of each sentence token output for the type of slot
# (5 classes - numSlots: B-Timer, I-Timer, B-Duration, I-Duration, O)

# The hidden_size in DistilBERT is 768 (embedding size)

# The intent classifier is modeled with a linear layer (MLP) of size (768, numIntents)
# The slot classifier is modeled with a linear layer (MLP) of size (768, numSlots)

# The calculated error (loss) is with cross-entropy
# Weight adjustment based on the loss is ONLY on the subset of tokens that are not None, nor -100

import torch
import torch.nn as nn

class JointIntentSlotModel(nn.Module):
    def __init__(self, num_intents, num_slots):
        super().__init__()

        self.encoder = AutoModel.from_pretrained("distilbert-base-uncased")

        hidden_size = self.encoder.config.hidden_size  # 768

        # Intent head (sentence-level)
        self.intent_classifier = nn.Linear(hidden_size, num_intents)

        # We can have a more complex MLP between the CLS head and the intent
        # self.intent_classifier = nn.Sequential(
        #  nn.Linear(768, 256),
        #  nn.ReLU(),
        #  nn.Dropout(0.1),
        #  nn.Linear(256, num_intents)
        # )

        # Slot head (token-level)
        self.slot_classifier = nn.Linear(hidden_size, num_slots)

        # We could also have a more complex MLP between the tokens and the slots
        # self.slot_classifier = nn.Sequential(
        #    nn.Linear(768, 256),
        #    nn.ReLU(),
        #    nn.Dropout(0.1),
        #    nn.Linear(256, num_slots)
        #   )

    # forward is when the activation goes from the input to the output
    # we can then calculate a loss between the output and the expected information
    def forward(self, input_ids, attention_mask,
                intent_labels=None,
                slot_labels=None):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state
        cls_output = sequence_output[:, 0]   # CLS token

        intent_logits = self.intent_classifier(cls_output)
        slot_logits   = self.slot_classifier(sequence_output)

        loss = None

        if intent_labels is not None and slot_labels is not None:

            intent_loss_fn = nn.CrossEntropyLoss()
            slot_loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)

            intent_loss = intent_loss_fn(
                intent_logits,
                intent_labels
            )

            slot_loss = slot_loss_fn(
                slot_logits.view(-1, slot_logits.shape[-1]),
                slot_labels.view(-1)
            )

            loss = intent_loss + slot_loss

        return {
            "loss": loss,
            "intent_logits": intent_logits,
            "slot_logits": slot_logits
        }

In [ ]:
# The finetuning model is instantiated with the proper set of intents and slots
# as that determines the architecture

num_intents = len(intent2id)
num_slots   = len(slot2id)

model = JointIntentSlotModel(num_intents, num_slots)

# Freeze the encoder (DistilBERT) - the two lines below (if False) tell the model that gradient descent is not necessary
# for the encoder's parameters, so those parameters will not be updated
# for param in model.encoder.parameters():
#   param.requires_grad = False

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# PyTorch has a method DataLoader requiring the dataset to be in a particular format
# batch_size is to do weight adjustment after a batch (instead of after each example)
# shuffle is to show the examples in random order when training (not always in same order)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8)

In [ ]:
# The actual training of the model in PyTorch

from torch.optim import AdamW

# Check if GPU available, if not use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# The optimizer is what will calculate how to adjust the weights based on the loss
# Learning rate is very small
optimizer = AdamW(model.parameters(), lr=5e-5)

# epochs is the number of times the training data is revisited
epochs = 5

for epoch in range(epochs):

    # sets the model in training mode
    model.train()
    total_loss = 0

    # cumulate the loss over batch_size before doing weight adjustments
    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        intent_labels = batch["intent_label"].to(device)
        slot_labels = batch["slot_labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            intent_labels=intent_labels,
            slot_labels=slot_labels
        )

        loss = outputs["loss"]

        optimizer.zero_grad()   # clear previous gradients
        loss.backward()         # propagate loss with backpropagation
        optimizer.step()        # update the model's parameters

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.3f}")

/tmp/ipykernel_25327/2216685523.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
/tmp/ipykernel_25327/2216685523.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),


Epoch 1 Loss: 194.364
Epoch 2 Loss: 37.455
Epoch 3 Loss: 15.165
Epoch 4 Loss: 7.259
Epoch 5 Loss: 5.661


In [ ]:
EXPORT_DIR = "/content/drive/MyDrive/project_dataset_audio/saved_intent_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(EXPORT_DIR, "intent_model.pt"))
tokenizer.save_pretrained(EXPORT_DIR)

('/content/drive/MyDrive/project_dataset_audio/saved_intent_model/tokenizer_config.json',
 '/content/drive/MyDrive/project_dataset_audio/saved_intent_model/tokenizer.json')

In [ ]:
import json

with open(os.path.join(EXPORT_DIR, "intent2id.json"), "w") as f:
    json.dump(intent2id, f, indent=4)

with open(os.path.join(EXPORT_DIR, "slot2id.json"), "w") as f:
    json.dump(slot2id, f, indent=4)


config = {
    "num_intents": len(intent2id),
    "num_slots": len(slot2id),
    "max_length": 16
}

with open(os.path.join(EXPORT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=4)

In [ ]:
# --------------------------
# Intent evaluation only
# --------------------------
from sklearn.metrics import classification_report
def evaluate_intent(model, data_loader, id2intent, device):
    model.eval()
    intent_true = []
    intent_pred = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            intent_labels = batch["intent_label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs["intent_logits"], dim=1)

            intent_true.extend(intent_labels.cpu().tolist())
            intent_pred.extend(preds.cpu().tolist())

    # Map IDs to string names
    true_names = [id2intent[i] for i in intent_true]
    pred_names = [id2intent[i] for i in intent_pred]

    acc = sum(t==p for t, p in zip(true_names, pred_names)) / len(true_names)

    print("Intent Accuracy:", acc)
    print("\nIntent Classification Report:")
    print(classification_report(true_names, pred_names))

    print("\nMisclassified Intents:")
    for i, (t, p) in enumerate(zip(true_names, pred_names)):
        if t != p:
            print(f"Example {i}: True: {t}, Predicted: {p}")

    return acc


In [ ]:
intent_acc = evaluate_intent(model, val_loader, id2intent, device)
print("Returned Intent Accuracy:", intent_acc)

In [ ]:
#Evaluation-slot only
# Python library for evaluating sequence labeling
# It handles BIO format and computes metrics (Precision/Recall/F1) by entity type (see example)

!pip install seqeval

In [ ]:
# --------------------------
# Slot evaluation only
# --------------------------
from seqeval.metrics import classification_report as seq_classification_report
from seqeval.metrics import f1_score as seq_f1_score
def evaluate_slots(model, data_loader, id2slot, device):
    model.eval()
    slot_true = []
    slot_pred = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            slot_labels = batch["slot_labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            slot_logits = outputs["slot_logits"]
            preds = torch.argmax(slot_logits, dim=2)

            for true_seq, pred_seq in zip(slot_labels, preds):
                true_labels = []
                pred_labels = []
                for t, p in zip(true_seq, pred_seq):
                    if t.item() != -100:
                        true_labels.append(id2slot[t.item()])
                        pred_labels.append(id2slot[p.item()])
                slot_true.append(true_labels)
                slot_pred.append(pred_labels)

    print("\nSlot Classification Report:")
    print(seq_classification_report(slot_true, slot_pred))
    f1 = seq_f1_score(slot_true, slot_pred)
    print("Slot F1 Score:", f1)
    return f1

In [ ]:
slot_f1 = evaluate_slots(model, val_loader, id2slot, device)
print("Returned Slot F1:", slot_f1)

/tmp/ipykernel_1937/2216685523.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
/tmp/ipykernel_1937/2216685523.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),



Slot Classification Report:
              precision    recall  f1-score   support

       ANAME       0.83      0.71      0.77         7
       BNAME       0.94      0.94      0.94        16
         DAY       0.83      1.00      0.91         5
    DURATION       1.00      1.00      1.00        10
    LOCATION       1.00      1.00      1.00         5
        NAME       0.67      0.50      0.57         4

   micro avg       0.91      0.89      0.90        47
   macro avg       0.88      0.86      0.86        47
weighted avg       0.91      0.89      0.90        47

Slot F1 Score: 0.9032258064516129
Returned Slot F1: 0.9032258064516129


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: O-BNAME seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: O-NAME seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


In [ ]:
def predict_from_text(text, model, tokenizer, id2intent, id2slot, device):
    """
    Takes a raw transcribed sentence (from Whisper or typed bypass)
    and returns predicted intent + slots.
    """
    model.eval()

    # Tokenize the input text
    encoding = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=16
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

    #  Predict
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Intent prediction + confidence
        intent_probs = torch.softmax(outputs["intent_logits"], dim=1)
        intent_pred_idx = torch.argmax(intent_probs, dim=1).item()
        pred_intent = id2intent[intent_pred_idx]
        intent_conf = intent_probs[0, intent_pred_idx].item()

        # Slot predictions
        slot_preds_idx = torch.argmax(outputs["slot_logits"], dim=2)[0]
        pred_slots = [id2slot[int(p)] for p in slot_preds_idx]

    # Extract slot values from text
    word_ids = encoding.word_ids(batch_index=0)
    extracted_slots = {}
    current_slot = ""
    current_type = None

    for token, label in zip(tokens, pred_slots):

        #Skip special tokens
        if token in ["[CLS]", "[SEP]"]:
            continue

        if label.startswith("B-"):
            # save previous entity
            if current_slot:
                extracted_slots[current_type] = current_slot

            current_type = label.split("-")[1]
            current_slot = token if not token.startswith("##") else token[2:]

        elif label.startswith("I-") and current_type:
            if token.startswith("##"):
                current_slot += token[2:]   # merge subword
            else:
                current_slot += " " + token

        else:
            if current_slot and current_type:
              extracted_slots[current_type] = current_slot

    # add last entity
    if current_slot:
        extracted_slots[current_type] = current_slot

    # ──
    #Print results
    print(f"\nInput text : {text}")
    print(f"Tokens     : {tokens}")
    print(f"Pred slots : {pred_slots}")
    print(f"Pred intent: {pred_intent} (conf={intent_conf:.3f})")
    print(f"Extracted slots: {extracted_slots}")

    return {
        "intent": pred_intent,
        "confidence": intent_conf,
        "slots": extracted_slots,
        "raw_slot_labels": pred_slots,
        "tokens": tokens
    }

In [ ]:
result = predict_from_text(speech["text"], model, tokenizer, id2intent, id2slot, device)
print(result["intent"])
result2 = predict_from_text(speech2["text"], model, tokenizer, id2intent, id2slot, device)
print(result2["intent"])

result3 = predict_from_text(speech3["text"], model, tokenizer, id2intent, id2slot, device)
print(result3["intent"])
result4 = predict_from_text(speech4["text"], model, tokenizer, id2intent, id2slot, device)
print(result4["intent"])
result5 = predict_from_text(speech5["text"], model, tokenizer, id2intent, id2slot, device)
print(result5["intent"])
result6 = predict_from_text(speech6["text"], model, tokenizer, id2intent, id2slot, device)
print(result6["intent"])
result7 = predict_from_text(speech7["text"], model, tokenizer, id2intent, id2slot, device)
print(result7["intent"])
result8 = predict_from_text(speech8["text"], model, tokenizer, id2intent, id2slot, device)
print(result8["intent"])

result9 = predict_from_text(speech9["text"], model, tokenizer, id2intent, id2slot, device)
print(result9["intent"])


Input text :  Who wrote Twilight?
Tokens     : ['[CLS]', 'who', 'wrote', 'twilight', '?', '[SEP]']
Pred slots : ['O', 'O', 'O', 'B-BNAME', 'O', 'B-BNAME']
Pred intent: GetAuthor (conf=0.994)
Extracted slots: {'BNAME': 'twilight'}
GetAuthor

Input text :  Open Harry Potter Book
Tokens     : ['[CLS]', 'open', 'harry', 'potter', 'book', '[SEP]']
Pred slots : ['O-BNAME', 'O', 'B-BNAME', 'I-BNAME', 'O', 'B-BNAME']
Pred intent: OpenBook (conf=0.938)
Extracted slots: {'BNAME': 'harry potter'}
OpenBook

Input text :  When was Harry Potter published?
Tokens     : ['[CLS]', 'when', 'was', 'harry', 'potter', 'published', '?', '[SEP]']
Pred slots : ['O', 'O', 'O', 'B-BNAME', 'I-BNAME', 'O', 'O', 'B-BNAME']
Pred intent: GetPublishingYear (conf=0.989)
Extracted slots: {'BNAME': 'harry potter'}
GetPublishingYear

Input text :  Get books by order Leo Tolstoy
Tokens     : ['[CLS]', 'get', 'books', 'by', 'order', 'leo', 'to', '##ls', '##toy', '[SEP]']
Pred slots : ['B-NAME', 'O', 'O', 'O', 'O', 'B-NAME

In [ ]:
import requests
import random
from datetime import datetime, timedelta

ereader_state = {
    "current_book": None,
    "current_page": 0,
    "reading_session": False,
    "dark_mode": False
}

weather_code_map = {
    0: "clear sky",
    1: "mainly clear",
    2: "partly cloudy",
    3: "overcast",
    45: "fog",
    48: "depositing rime fog",
    51: "light drizzle",
    53: "drizzle",
    55: "dense drizzle",
    61: "rain",
    63: "moderate rain",
    65: "heavy rain",
    71: "light snow",
    73: "snow",
    75: "heavy snow",
    80: "rain showers",
    81: "moderate rain showers",
    82: "violent rain showers",
    95: "thunderstorm",
    99: "thunderstorm with hail"
}

In [ ]:
def get_coordinates(city, user_lat=45.4215, user_lon=-75.6972):
  url = "https://geocoding-api.open-meteo.com/v1/search"
  params = {
      "name": city,
      "count": 10,
      "language": "en",
      "format": "json"
  }

  response = requests.get(url, params=params)
  if response.status_code != 200:
    return None

  data = response.json()
  if "results" not in data or len(data["results"]) == 0:
    return None

  # choose nearest result if multiple are returned
  def dist_sq(result):
    return (result["latitude"] - user_lat) ** 2 + (result["longitude"] - user_lon) ** 2

  best = min(data["results"], key=dist_sq)
  return best["latitude"], best["longitude"]

In [ ]:
def call_weather_api(lat, lon):
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
      "latitude": lat,
      "longitude": lon,
      "current": "temperature_2m,weather_code,wind_speed_10m",
      "daily": "weather_code,temperature_2m_max,temperature_2m_min",
      "timezone": "auto",
      "forecast_days": 3
  }

  response = requests.get(url, params=params)
  #print("STATUS:", response.status_code)  # DEBUG
  if response.status_code != 200:
    return None

  return response.json()

In [ ]:
def extract_weather_info(weather_data):
  if weather_data is None:
    return None

  current = weather_data["current"]

  info = {
      "temperature": current["temperature_2m"],
      "windspeed": current["wind_speed_10m"],
      "weathercode": current["weather_code"],
      "daily_forecast": []
  }

  daily = weather_data.get("daily", {})
  dates = daily.get("time", [])
  highs = daily.get("temperature_2m_max", [])
  lows = daily.get("temperature_2m_min", [])
  codes = daily.get("weathercode", [])

  for i in range(min(len(dates), len(highs), len(lows), len(codes))):
    info["daily_forecast"].append({
        "date": dates[i],
        "high": highs[i],
        "low": lows[i],
        "weathercode": codes[i]
      })

  return info

In [ ]:
def generate_weather_answer(city, info):
  description = weather_code_map.get(info["weathercode"], "unknown conditions")

  answer1 = (
      f"The current weather in {city} is {description}. "
      f"The temperature is {info['temperature']}°C with wind speed of {info['windspeed']} km/h."
    )
  answer2 = (
      f"In {city}, it is currently {description}. "
      f"The temperature is {info['temperature']}°C and the wind speed is {info['windspeed']} km/h."
    )
  answer3 = (
      f"{city} is currently experiencing {description}, "
      f"with a temperature of {info['temperature']}°C and wind speed of {info['windspeed']} km/h."
      )

  return random.choice([answer1, answer2, answer3])

In [ ]:
## OPEN LIBRARY HELPERS

def call_open_library_search_api(title=None, author=None):
  url = "https://openlibrary.org/search.json"
  params = {}

  if title:
    params["title"] = title
  if author:
    params["author"] = author

  response = requests.get(url, params=params)
  if response.status_code != 200:
    return None

  return response.json()

In [ ]:
def extract_book_info(data):
  if data is None or "docs" not in data or len(data["docs"]) == 0:
    return None

  first = data["docs"][0]

  return {
      "title": first.get("title", "Unknown title"),
      "author_name": first.get("author_name", ["Unknown author"]),
      "first_publish_year": first.get("first_publish_year", "Unknown year"),
      "docs": data["docs"]
  }

In [ ]:
def generate_book_answer(intent, slots, info):
  if intent == "GetBook":
    title = info["title"]
    author = info["author_name"][0]

    answer1 = f"I found {title} by {author}."
    answer2 = f"The book I found is {title}, written by {author}."
    answer3 = f"I found a result for that request: {title} by {author}."
    return random.choice([answer1, answer2, answer3])
  elif intent == "GetAuthor":
    requested_title = slots.get("book_title", info["title"])
    author = info["author_name"][0]

    answer1 = f"The author of {requested_title} is {author}."
    answer2 = f"{requested_title} was written by {author}."
    answer3 = f"The listed author for {requested_title} is {author}."
    return random.choice([answer1, answer2, answer3])
  elif intent == "GetPublishingYear":
    requested_title = slots.get("book_title", info["title"])
    year = info["first_publish_year"]
    answer1 = f"{requested_title} was first published in {year}."
    answer2 = f"The first publish year listed for {requested_title} is {year}."
    answer3 = f"{requested_title} appears to have first been published in {year}."
    return random.choice([answer1, answer2, answer3])
  elif intent == "GetBooksByAuthor":
    author = slots.get("author_name", "that author")
    titles = []

    for doc in info["docs"][:5]:
      if "title" in doc:
        titles.append(doc["title"])

    if len(titles) == 0:
      return f"I could not find any books by {author}."

    titles_text = ", ".join(titles)

    answer1 = f"Some books by {author} include {titles_text}."
    answer2 = f"I found these books by {author}: {titles_text}."
    answer3 = f"{author} has books such as {titles_text}."
    return random.choice([answer1, answer2, answer3])

  return "Sorry, I could not generate a book answer."

In [ ]:
def generate_basic_answer(intent, slots=None):
  slots = slots or {}

  if intent == "Greeting":
    return random.choice([
        "Hello! How can I help you today?",
        "Hi there! What would you like me to do?",
        "Hey! I'm ready to help."
    ])
  elif intent == "Goodbye":
    return random.choice([
        "Goodbye!",
        "See you later!",
        "Bye! Have a great day!"
    ])
  elif intent in ["SetTimer", "SetAlarm"]:
    duration = slots.get("DURATION", "the requested amount of time")
    return random.choice([
        f"Timer set for {duration}.",
        f"Alright, I started a timer for {duration}.",
        f"Your timer for {duration} is now running."
      ])
  return "Sorry, I could not generate a response."

In [ ]:
def fulfill_ereader_control(intent, slots):
  if intent == "OpenBook":
    title = slots.get("title") or slots.get("BNAME") or slots.get("book_title") or "Unknown Book"
    ereader_state["current_book"] = title
    ereader_state["current_page"] = 1
    ereader_state["reading_session"] = True

    return random.choice([
        f"Opened {title}. You are now on page 1.",
        f"{title} is now open. Current page: 1.",
        f"I opened {title} for you. You are on page 1."
      ])

  elif intent == "NextPage":
    if ereader_state["reading_session"] is False or ereader_state["current_book"] is None:
      return "You need to open a book first."

    ereader_state["current_page"] += 1
    page = ereader_state["current_page"]

    return random.choice([
          f"Turned to the next page. You are now on page {page}.",
          f"Page advanced. Current page is {page}.",
          f"You are now on page {page}."
        ])

  elif intent == "EnableDarkMode":
    ereader_state["dark_mode"] = True

    return random.choice([
        "Dark mode has been enabled.",
        "I turned on dark mode.",
        "The e-reader is now in dark mode."
      ])

  return "Unknown e-reader control request."

In [ ]:
def fulfill_intent(request):
  intent = request["intent"]
  slots = request.get("slots", {})

  ## Greetings, Goodbye, Timer
  if intent in ["Greeting", "Goodbye", "SetTimer", "SetAlarm"]:
    return generate_basic_answer(intent, slots)

  # Weather
  elif intent == "AskForWeather":
    city = slots.get("LOCATION") or slots.get("city")

    if not city or city.strip() == "":
      city = "Ottawa"

    coords = get_coordinates(city)
    if coords is None:
      return "Sorry, I could not find that city"

    lat, lon = coords
    weather_data = call_weather_api(lat, lon)
    if weather_data is None:
      return "Sorry, I could not retrieve the weather right now"

    info = extract_weather_info(weather_data)
    return generate_weather_answer(city, info)

  # Book
  elif intent in ["GetBook", "GetAuthor", "GetPublishingYear"]:
    book_title = (
        slots.get("book_title") or
        slots.get("title") or
        slots.get("book_name") or
        slots.get("BNAME")
    )

    if not book_title:
      return "Sorry, I need a book title for that request"

    data = call_open_library_search_api(title=book_title)
    info = extract_book_info(data)

    if info is None:
      return "Sorry, I could not find that book."

    return generate_book_answer(intent, slots, info)

  elif intent == "GetBooksByAuthor":
    author_name = (slots.get("author_name") or slots.get("NAME"))

    if not author_name:
      return "Sorry, I need an author name for that request"

    data = call_open_library_search_api(author=author_name)
    info = extract_book_info(data)

    if info is None:
      return "Sorry, I could not find any books by that author."

    return generate_book_answer(intent, slots, info)

  # e-reader control
  elif intent in ["OpenBook", "NextPage", "EnableDarkMode"]:
    return fulfill_ereader_control(intent, slots)

  # Out of scope
  elif intent == "OOS":
    return "Sorry, that request is outside the scope of this assistant."

  return "Sorry, I do not know how to fulfill that intent."

In [ ]:
# Testing
print(fulfill_intent({"intent": "Greeting", "slots": {}}))
print(fulfill_intent({"intent": "Goodbye", "slots": {}}))
print(fulfill_intent({"intent": "SetAlarm", "slots": {"duration": "2 minutes"}}))
print(fulfill_intent({"intent": "AskForWeather", "slots": {"location": "Ottawa"}}))
print(fulfill_intent({"intent": "GetAuthor", "slots": {"book_title": "Nineteen eighty-four"}}))
print(fulfill_intent({"intent": "GetPublishingYear", "slots": {"book_title": "Twilight"}}))
print(fulfill_intent({"intent": "OpenBook", "slots": {"title": "The Hobbit"}}))
print(fulfill_intent({"intent": "NextPage", "slots": {}}))
print(fulfill_intent({"intent": "EnableDarkMode", "slots": {}}))

Hey! I'm ready to help.
Goodbye!
Timer set for the requested amount of time.
Ottawa is currently experiencing rain, with a temperature of 14.3°C and wind speed of 21.4 km/h.
The listed author for Nineteen eighty-four is George Orwell.
Twilight was first published in 2005.
Opened The Hobbit. You are now on page 1.
Page advanced. Current page is 2.
Dark mode has been enabled.


In [ ]:
print(fulfill_intent({"intent": result["intent"], "slots":result["slots"]}))
print(fulfill_intent({"intent": result2["intent"], "slots":result2["slots"]}))
print(fulfill_intent({"intent": result3["intent"], "slots":result3["slots"]}))

print(fulfill_intent({"intent": result4["intent"], "slots":result4["slots"]}))
print(fulfill_intent({"intent": result5["intent"], "slots":result5["slots"]}))

print(fulfill_intent({"intent": result6["intent"], "slots":result6["slots"]}))
print(fulfill_intent({"intent": result7["intent"], "slots":result7["slots"]}))

print(fulfill_intent({"intent": result8["intent"], "slots":result8["slots"]}))

print(fulfill_intent({"intent": result9["intent"], "slots":result9["slots"]}))

The listed author for Twilight is Stephenie Meyer.
I opened harry potter for you. You are on page 1.
Harry Potter and the Deathly Hallows appears to have first been published in 2007.


SSLError: HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?author=leo+tolstoy (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
)

def build_qwen_prompt(request):
    intent = request["intent"]
    slots = request.get("slots", {})

    if intent == "Greeting":
        task = "Generate a short greeting response."
    elif intent == "Goodbye":
        task = "Generate a short goodbye response."
    elif intent in ["SetTimer", "SetAlarm"]:
        duration = slots.get("duration", "the requested time")
        task = f"Generate a short response confirming that a timer was set for {duration}."
    elif intent == "GetAuthor":
        book_title = slots.get("book_title", "")
        task = (
            f"Generate a short response giving the author of the book titled '{book_title}'. "
            f"Do not ask follow-up questions."
        )
    elif intent == "GetPublishingYear":
        book_title = slots.get("book_title", "")
        task = (
            f"Generate a short response giving the first publishing year of the book titled '{book_title}'. "
            f"Do not ask follow-up questions."
        )
    elif intent == "AskForWeather":
        city = slots.get("location") or slots.get("city") or "Ottawa"
        task = (
            f"Generate a short weather response for {city}. "
            f"Do not ask follow-up questions."
        )
    elif intent == "OpenBook":
        title = slots.get("title") or slots.get("book_title") or "Unknown Book"
        task = f"Generate a short response confirming that the book '{title}' was opened on page 1."
    elif intent == "NextPage":
        task = "Generate a short response confirming that the page was advanced."
    elif intent == "EnableDarkMode":
        task = "Generate a short response confirming that dark mode was enabled."
    else:
        task = "Generate a short helpful assistant response."

    return (
        "You are a voice assistant.\n"
        "Respond in one short sentence.\n"
        "Do not ask follow-up questions.\n"
        "Do not mention intents or slots.\n"
        f"{task}"
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def generate_qwen_answer(request, max_new_tokens=256):
  prompt = build_qwen_prompt(request)

  messages = [
      {"role": "system", "content": "You are a helpful assistant."},
       {"role": "user", "content": prompt}
  ]

  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(text, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

  generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
  answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
  return answer

In [ ]:
def compare_answer_generation(request):
    template_answer = fulfill_intent(request)
    qwen_answer = generate_qwen_answer(request)

    return {
        "request": request,
        "template_answer": template_answer,
        "qwen_answer": qwen_answer
    }

In [ ]:
print(compare_answer_generation({
    "intent": "Greeting",
    "slots": {}
}))

print(compare_answer_generation({
    "intent": "GetAuthor",
    "slots": {"book_title": "Nineteen eighty-four"}
}))

print(compare_answer_generation({
    "intent": "GetPublishingYear",
    "slots": {"book_title": "Twilight"}
}))

print(compare_answer_generation({
    "intent": "AskForWeather",
    "slots": {"location": "Montreal"}
}))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{'request': {'intent': 'Greeting', 'slots': {}}, 'template_answer': 'Hi there! What would you like me to do?', 'qwen_answer': 'Hello! How can I assist you today?'}
{'request': {'intent': 'GetAuthor', 'slots': {'book_title': 'Nineteen eighty-four'}}, 'template_answer': 'The author of Nineteen eighty-four is George Orwell.', 'qwen_answer': 'George Orwell wrote "Nineteen Eighty-Four".'}
{'request': {'intent': 'GetPublishingYear', 'slots': {'book_title': 'Twilight'}}, 'template_answer': 'Twilight appears to have first been published in 2005.', 'qwen_answer': "'Twilight' was published in 2005."}
{'request': {'intent': 'AskForWeather', 'slots': {'location': 'Montreal'}}, 'template_answer': 'The current weather in Montreal is light drizzle. The temperature is 9.8°C with wind speed of 11.3 km/h.', 'qwen_answer': 'The current temperature in Montreal is 10°C with partly cloudy skies.'}


In [ ]:
def get_visual_display():
    return {
        "current_book": ereader_state["current_book"],
        "current_page": ereader_state["current_page"],
        "reading_session": ereader_state["reading_session"],
        "dark_mode": ereader_state["dark_mode"]
    }

Text-to-speech  
• Input: Generated response in NL   
• Task: Transform the text into speech.   
• Output State: Speech   

In [ ]:
!pip install edge-tts

In [ ]:
import edge_tts
import asyncio
import os
from IPython.display import Audio, display

In [ ]:
# simple TTS
async def generate_tts(text, voice="en-CA-ClaraNeural", output_file="output.mp3"):
    communicate = edge_tts.Communicate(text=text, voice=voice)
    await communicate.save(output_file)
    return output_file

In [ ]:
async def generate_emotional_tts(
    text,
    strategy,
    voice="en-CA-ClaraNeural",
    output_file="response.mp3"
):
    strategy_to_prosody = {
        "reinforce_positive": {
            "rate": "+12%",
            "pitch": "+18Hz",
            "volume": "+8%"
        },
        "supportive": {
            "rate": "-18%",
            "pitch": "-8Hz",
            "volume": "-5%"
        },
        "calm_down": {
            "rate": "-22%",
            "pitch": "-12Hz",
            "volume": "-10%"
        },
        "reassuring": {
            "rate": "-10%",
            "pitch": "-5Hz",
            "volume": "+0%"
        },
        "neutralize": {
            "rate": "-5%",
            "pitch": "-2Hz",
            "volume": "+0%"
        },
        "informative": {
            "rate": "+0%",
            "pitch": "+0Hz",
            "volume": "+0%"
        },
        "neutral_response": {
            "rate": "+0%",
            "pitch": "+0Hz",
            "volume": "+0%"
        },
        "clarify_intent": {
            "rate": "-8%",
            "pitch": "+5Hz",
            "volume": "+0%"
        },
        "gentle_probe": {
            "rate": "-15%",
            "pitch": "-3Hz",
            "volume": "-3%"
        }
    }

    prosody = strategy_to_prosody.get(strategy, strategy_to_prosody["neutral_response"])

    communicate = edge_tts.Communicate(
        text=text,
        voice=voice,
        rate=prosody["rate"],
        pitch=prosody["pitch"],
        volume=prosody["volume"]
    )

    await communicate.save(output_file)
    return output_file

In [ ]:
# Helper: choose speaking style based on request intent
def choose_tts_strategy(request):
    intent = request["intent"]

    if intent in ["Greeting", "Goodbye"]:
        return "reinforce_positive"

    elif intent in ["SetTimer", "SetAlarm", "OpenBook", "NextPage", "EnableDarkMode"]:
        return "reassuring"

    elif intent in ["AskForWeather", "GetAuthor", "GetPublishingYear", "GetBook", "GetBooksByAuthor"]:
        return "informative"

    elif intent == "OOS":
        return "neutralize"

    return "neutral_response"

In [ ]:
# Takes the NL answer from Module 6 and converts it to speech
async def speak_generated_answer(
    request,
    answer_source="template",
    voice="en-CA-ClaraNeural",
    output_file="assistant_response.mp3"
):
    # Step 1: Generate the natural language answer
    if answer_source == "template":
        response_text = fulfill_intent(request)

    elif answer_source == "qwen":
        response_text = generate_qwen_answer(request)

    else:
        raise ValueError("answer_source must be 'template' or 'qwen'")

    # Step 2: Pick TTS style
    strategy = choose_tts_strategy(request)

    # Step 3: Convert answer text to speech
    audio_file = await generate_emotional_tts(
        text=response_text,
        strategy=strategy,
        voice=voice,
        output_file=output_file
    )

    return {
        "request": request,
        "response_text": response_text,
        "strategy": strategy,
        "audio_file": audio_file
    }

In [ ]:
# Wrapper for both displaying text and playing audio
async def answer_and_speak(request, answer_source="template", voice="en-CA-ClaraNeural"):
    result = await speak_generated_answer(
        request=request,
        answer_source=answer_source,
        voice=voice,
        output_file="final_response.mp3"
    )

    visual_state = get_visual_display()

    return {
        "request": request,
        "response_text": result["response_text"],
        "audio_file": result["audio_file"],
        "visual_state": visual_state
    }

In [ ]:
# Testing
template_result = await speak_generated_answer(
    {
        "intent": "OpenBook",
        "slots": {"title": "The Hobbit"}
    },
    answer_source="template",
    voice="en-CA-ClaraNeural",
    output_file="template_openbook.mp3"
)

print("Template answer:", template_result["response_text"])
print("TTS strategy:", template_result["strategy"])
display(Audio(template_result["audio_file"]))

Template answer: Opened The Hobbit. You are now on page 1.
TTS strategy: reassuring


In [ ]:
# Test 2
qwen_result = await speak_generated_answer(
    {
        "intent": "OpenBook",
        "slots": {"title": "The Hobbit"}
    },
    answer_source="qwen",
    voice="en-CA-LiamNeural",
    output_file="qwen_openbook.mp3"
)

print("Qwen answer:", qwen_result["response_text"])
print("TTS strategy:", qwen_result["strategy"])
display(Audio(qwen_result["audio_file"]))

Qwen answer: Yes, 'The Hobbit' has been opened to page 1.
TTS strategy: reassuring
